# «Бот» V4: несколько локальных TTS и многоязычный акустический encoder, две GPU

**Результат:** компактный бинарный детектор, который получает звук и выдаёт
`probability` / `wake`. Он не распознаёт команды и не возвращает текст.
По умолчанию положительное обращение — отдельно произнесённое **«бот» / “bot”**.
Для нового слова измените параметры ниже и обучите новую модель.

В ноутбуке уже есть тексты, генератор нейросетевой речи, аугментации, модели,
обучение, калибровка и экспорт. Внешний Dataset, исходники бота и API-ключи не нужны.
Датасет создаётся **при выполнении**, готовые WAV-файлы не спрятаны внутри файла.

В Kaggle выберите **Accelerator → GPU T4 ×2**, включите **Internet**, импортируйте
этот `.ipynb` и запускайте ячейки сверху вниз / Save & Run All.
Генерация запускает два независимых процесса TTS, обучение — два процесса DDP.
Программа проверяет число GPU; одна карта не будет молча выдана за две.

V4 использует Silero v4/v5.5, Piper, MMS, Kokoro, реальные отрицательные слова,
многоязычный encoder Whisper Tiny без decoder, баланс русского языка, hard-example mining, EMA,
два запуска с разными seed и выбор ансамбля только по validation.
Максимальная точность не гарантируется количеством эпох или синтетических файлов.
Test оценивает перенос на другие TTS-голоса,
но не доказывает качество на реальных микрофонах, акцентах и музыке Discord.
Модель пока не подключается к боту автоматически: будущая схема — локальный
wake-detector → сигнал → Groq STT только для команды.


## 1. Параметры

`STANDALONE_ONLY=True` соответствует текущему сценарию: имя → сигнал → команда.
Упоминания вроде «этот бот» — отрицательные примеры. При `False` положительные
примеры включают короткие фразы с обращением; это другая задача, нужен новый запуск.
Один акустический детектор не определяет намерение говорящего по идентичному звуку.

`USE_ENGLISH=True` добавляет английское произношение “bot” и голоса всех движков.
При смене русского слова обновите английский вариант либо отключите английскую часть.
Созвучные слова дополнительно настройте в `wake_data.py`, если выбрано другое имя.
`SMOKE=True` — проверка конвейера на небольшом наборе; экспорт помечается SMOKE-ONLY.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, hashlib, platform

WAKE_WORD_RU = "бот"
WAKE_WORD_EN = "bot"
USE_ENGLISH = True
STANDALONE_ONLY = True
REQUIRE_TWO_GPUS = True
SMOKE = False
SEED = 20261005
POSITIVE_PER_VOICE = 300
NEGATIVE_PER_VOICE = 600
EVAL_POSITIVE_PER_VOICE = 200
EVAL_NEGATIVE_PER_VOICE = 800
RUSSIAN_MULTIPLIER = 2
TTS_ENGINES = ['silero', 'piper', 'mms', 'kokoro']
SILERO_RU_VERSIONS = ['v5_5_ru', 'v4_ru']
GENDER_BALANCED_RU = True
POSITIVE_TTS_VARIANTS = 6
REAL_SPEECH_SOURCE = 'full'  # 'mini' is smaller; full download is about 2.4 GB
REAL_SPEECH_WINDOWS = 12000  # 0 disables real negative speech
REAL_WAKE_MANIFEST = None  # Optional reviewed real recordings; schema is described below
INDEPENDENT_PITCH = True
NOISE_WINDOWS = 2400
EPOCHS = 100  # maximum, early stopping may finish sooner
PATIENCE = 18
MIN_EPOCHS = 25
ENSEMBLE_MEMBERS = 2
ARCHITECTURE = 'whisper_encoder_v4'  # alternatives: 'temporal_v2', 'speech_embedding_v3', 'legacy'; new WORK_DIR required
ENCODER_TRAIN_LAYERS = 2  # Whisper encoder only: 0..4, last layers, no speech decoder
ENCODER_LEARNING_RATE = 0.00001
BATCH_PER_GPU = 64
LEARNING_RATE = 0.0007
TARGET_FPR_PER_WINDOW = 0.005
RESUME_CHECKPOINT = None  # ensemble: path to previous checkpoints/ containing member-0/ and member-1/
INCLUDE_DATASET_IN_ZIP = False  # WAV + features remain in Kaggle Output either way

# Use a new directory for a different word, split or synthesis recipe.
WORK_DIR = Path('/kaggle/working/wakeword-bot-v4')
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not WAKE_WORD_RU.strip() or len(WAKE_WORD_RU.split()) != 1:
    raise ValueError('WAKE_WORD_RU must be one word')
if USE_ENGLISH and (not WAKE_WORD_EN.strip() or len(WAKE_WORD_EN.split()) != 1):
    raise ValueError('Set one English word, or USE_ENGLISH=False')
if not 0 <= TARGET_FPR_PER_WINDOW < 1:
    raise ValueError('TARGET_FPR_PER_WINDOW must be in [0, 1)')
CONFIG = dict(work_dir=str(WORK_DIR), wake_word_ru=WAKE_WORD_RU, wake_word_en=WAKE_WORD_EN,
              use_english=USE_ENGLISH, standalone_only=STANDALONE_ONLY, seed=SEED,
              positive_per_voice=8 if SMOKE else POSITIVE_PER_VOICE,
              negative_per_voice=12 if SMOKE else NEGATIVE_PER_VOICE,
              noise_windows=16 if SMOKE else NOISE_WINDOWS, epochs=2 if SMOKE else EPOCHS,
              batch_per_gpu=8 if SMOKE else BATCH_PER_GPU, learning_rate=LEARNING_RATE,
              target_fpr=TARGET_FPR_PER_WINDOW, resume_checkpoint=RESUME_CHECKPOINT,
              loader_workers=2, cpu_threads=2, mixed_precision=True, smoke=SMOKE)
CONFIG.update(recipe_version=4, tts_engines=TTS_ENGINES, silero_ru_versions=SILERO_RU_VERSIONS,
              gender_balanced_ru=GENDER_BALANCED_RU,
              eval_positive_per_voice=EVAL_POSITIVE_PER_VOICE, eval_negative_per_voice=EVAL_NEGATIVE_PER_VOICE,
              russian_multiplier=RUSSIAN_MULTIPLIER, positive_tts_variants=3 if SMOKE else POSITIVE_TTS_VARIANTS,
              real_speech_source='mini' if SMOKE else REAL_SPEECH_SOURCE,
              real_speech_windows=min(160, REAL_SPEECH_WINDOWS) if SMOKE else REAL_SPEECH_WINDOWS,
              real_wake_manifest=REAL_WAKE_MANIFEST,
              independent_pitch=INDEPENDENT_PITCH, architecture=ARCHITECTURE, balanced_sampling=True,
              encoder_train_layers=ENCODER_TRAIN_LAYERS, encoder_learning_rate=ENCODER_LEARNING_RATE,
              ema=True, preload_features=True, mining_interval=10, patience=PATIENCE, min_epochs=MIN_EPOCHS,
              ensemble_members=1 if SMOKE else ENSEMBLE_MEMBERS)
immutable = {key: CONFIG[key] for key in ['wake_word_ru', 'wake_word_en', 'use_english', 'standalone_only',
             'seed', 'positive_per_voice', 'negative_per_voice', 'noise_windows', 'smoke', 'recipe_version',
             'tts_engines', 'silero_ru_versions', 'gender_balanced_ru', 'eval_positive_per_voice', 'eval_negative_per_voice',
             'russian_multiplier', 'positive_tts_variants', 'real_speech_source', 'real_speech_windows', 'real_wake_manifest', 'independent_pitch', 'architecture']}
recipe = WORK_DIR / 'dataset_recipe.json'
if recipe.exists() and json.loads(recipe.read_text(encoding='utf-8')) != immutable:
    raise ValueError('WORK_DIR already contains a different dataset recipe; choose a new directory')
recipe.write_text(json.dumps(immutable, ensure_ascii=False, indent=2), encoding='utf-8')
CONFIG_FILE = WORK_DIR / 'run_config.json'
CONFIG_FILE.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
os.environ['OPENBLAS_NUM_THREADS'] = str(CONFIG['cpu_threads'])
os.environ['OMP_NUM_THREADS'] = str(CONFIG['cpu_threads'])
CODE_DIR = WORK_DIR / 'code'
CODE_DIR.mkdir(exist_ok=True)
print(f'Word: {WAKE_WORD_RU}; English: {WAKE_WORD_EN if USE_ENGLISH else "disabled"}; smoke: {SMOKE}')


## 2. Зависимости и обе GPU

Сохраняем установленную в Kaggle сборку PyTorch/CUDA: переустановка `torch`
и несовместимых `torchaudio` не нужна. Устанавливаются только вспомогательные пакеты.
FP16 включается на GPU, веса и экспорт остаются FP32.
В файле `environment.json` фиксируются фактические версии пакетов и названия GPU.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'numpy>=2.0.2,<3', 'scipy>=1.12,<2', 'matplotlib>=3.8,<4', 'onnx>=1.17,<2', 'onnxruntime>=1.20,<2',
                'piper-tts==1.8.0', 'kokoro-onnx==0.4.9', 'transformers==4.57.6', 'uroman==1.3.1.1',
                'librosa==0.11.0', 'huggingface_hub>=0.34,<1'], check=True)
import torch
GPU_COUNT = min(2, torch.cuda.device_count())
if not torch.cuda.is_available() or GPU_COUNT == 0:
    raise RuntimeError('Enable GPU T4 ×2 in Kaggle Settings')
if REQUIRE_TWO_GPUS and GPU_COUNT != 2:
    raise RuntimeError(f'Found {GPU_COUNT} GPU; select T4 ×2, or explicitly disable REQUIRE_TWO_GPUS')
import importlib.metadata
environment = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda,
                   gpus=[torch.cuda.get_device_name(i) for i in range(GPU_COUNT)],
                   packages={name: importlib.metadata.version(name) for name in ['numpy', 'scipy', 'matplotlib', 'onnx', 'onnxruntime',
                                                                               'piper-tts', 'kokoro-onnx', 'transformers', 'librosa']})
(WORK_DIR / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))


## 3. Встроенный код

Следующие восемь ячеек записывают полностью включённые в ноутбук модули в Output.
Для обычного запуска их редактировать не нужно; содержимое можно раскрыть.
Одна копия кода используется для обучения и экспорта frontend, чтобы избежать
несовпадения признаков. Вход: mono 16 кГц, окно 2 секунды. `temporal_v2`
использует log-mel 40 × 201, `speech_embedding_v3` — готовые акустические
признаки 96 × 16 из закреплённого encoder. Его две ONNX-модели скачиваются
автоматически и включаются в ZIP; ASR для детектора не нужен.
`whisper_encoder_v4` использует 80 × 200 log-mel и многоязычный encoder
Whisper Tiny с окном, сокращённым до двух секунд. Дообучаются последние
два encoder-слоя и бинарная голова, с разными learning rate. Decoder отсутствует.


In [ ]:
(CODE_DIR / 'wake_audio.py').write_text(r'''"""Shared NumPy frontend used for training and exported CPU inference."""
from pathlib import Path
import json
import wave
import numpy as np

SAMPLE_RATE = 16000
N_SAMPLES = 32000
N_FFT = 512
HOP = 160
N_MELS = 40
N_FRAMES = 1 + N_SAMPLES // HOP


def mel_filters():
    mel = lambda hz: 2595 * np.log10(1 + hz / 700)
    edges = 700 * (10 ** (np.linspace(mel(50), mel(7600), N_MELS + 2) / 2595) - 1)
    frequencies = np.fft.rfftfreq(N_FFT, 1 / SAMPLE_RATE)
    lower = (frequencies[None, :] - edges[:-2, None]) / (edges[1:-1] - edges[:-2])[:, None]
    upper = (edges[2:, None] - frequencies[None, :]) / (edges[2:] - edges[1:-1])[:, None]
    return np.maximum(0, np.minimum(lower, upper)).astype(np.float32)


MEL = mel_filters()
HANN = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(N_FFT) / N_FFT)).astype(np.float32)


def features(audio):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.shape != (N_SAMPLES,) or not np.isfinite(audio).all():
        raise ValueError('Expected exactly 32000 finite mono float32 samples at 16 kHz')
    padded = np.pad(audio, (N_FFT // 2, N_FFT // 2))
    frames = np.lib.stride_tricks.sliding_window_view(padded, N_FFT)[::HOP]
    power = np.abs(np.fft.rfft(frames * HANN, axis=-1)) ** 2
    logged = np.log(np.maximum(MEL @ power.T, 1e-6)).astype(np.float32)
    normalized = (logged - logged.mean()) / max(float(logged.std()), 1e-4)
    return normalized[None].astype(np.float32)


def read_wav(file):
    with wave.open(str(file), 'rb') as source:
        if source.getnchannels() != 1 or source.getsampwidth() != 2 or source.getframerate() != SAMPLE_RATE:
            raise ValueError('Expected mono PCM16 WAV at 16000 Hz')
        return np.frombuffer(source.readframes(source.getnframes()), dtype='<i2').astype(np.float32) / 32768


def write_wav(file, audio):
    with wave.open(str(file), 'wb') as destination:
        destination.setnchannels(1)
        destination.setsampwidth(2)
        destination.setframerate(SAMPLE_RATE)
        destination.writeframes((np.clip(audio, -1, 1) * 32767).astype('<i2').tobytes())


def right_aligned_window(audio):
    """Runtime contract: keep the most recent 2 seconds, left-pad short phrases."""
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)
    if len(audio) >= N_SAMPLES:
        return audio[-N_SAMPLES:].copy()
    return np.pad(audio, (N_SAMPLES - len(audio), 0))


class WakeDetector:
    def __init__(self, directory):
        import onnxruntime as ort
        directory = Path(directory)
        self.config = json.loads((directory / 'wake_config.json').read_text(encoding='utf-8'))
        if self.config['sample_rate'] != SAMPLE_RATE or self.config['window_samples'] != N_SAMPLES:
            raise ValueError('Incompatible frontend')
        import hashlib
        for name, expected in self.config.get('frontend_sha256', {}).items():
            if Path(name).name != name or hashlib.sha256((directory / name).read_bytes()).hexdigest() != expected:
                raise ValueError('Incompatible pretrained frontend hash')
        options = ort.SessionOptions()
        options.intra_op_num_threads = 2; options.inter_op_num_threads = 1
        self.session = ort.InferenceSession(str(directory / 'wake_model.onnx'), options, providers=['CPUExecutionProvider'])
        self.frontend = make_frontend(self.config, directory)

    def score(self, audio):
        tensor = self.frontend(right_aligned_window(audio))[None]
        return float(self.session.run(['probability'], {'log_mel': tensor})[0].reshape(-1)[0])

    def detect(self, audio):
        probability = self.score(audio)
        return {'wake': probability >= self.config['threshold'], 'probability': probability}


class SpeechEmbeddingFrontend:
    """Frozen Google speech features via the fixed openWakeWord v0.5.1 conversion.

    Full-window inference is used for both training and runtime; streaming buffers
    have different boundary behavior and are deliberately outside this contract.
    """
    def __init__(self, directory):
        import onnxruntime as ort
        directory = Path(directory)
        options = ort.SessionOptions()
        options.intra_op_num_threads = 2; options.inter_op_num_threads = 1
        self.mel = ort.InferenceSession(str(directory / 'melspectrogram.onnx'), options, providers=['CPUExecutionProvider'])
        self.embedding = ort.InferenceSession(str(directory / 'embedding_model.onnx'), options, providers=['CPUExecutionProvider'])

    def __call__(self, audio):
        audio = np.asarray(audio, dtype=np.float32)
        if audio.shape != (N_SAMPLES,) or not np.isfinite(audio).all():
            raise ValueError('Expected exactly 32000 finite mono samples')
        pcm = (np.clip(audio, -1, 1) * 32768).clip(-32768, 32767).astype(np.int16).astype(np.float32)[None]
        mel = self.mel.run(None, {'input': pcm})[0].reshape(-1, 32) / 10 + 2
        windows = np.stack([mel[start:start+76] for start in range(0, len(mel)-75, 8)])[..., None]
        value = self.embedding.run(None, {'input_1': windows.astype(np.float32)})[0].reshape(-1, 96)
        if value.shape != (16, 96) or not np.isfinite(value).all():
            raise ValueError(f'Unexpected embedding shape: {value.shape}')
        return value.T[None].astype(np.float32)


def make_frontend(config, directory):
    if config.get('architecture') == 'speech_embedding_v3':
        return SpeechEmbeddingFrontend(directory)
    if config.get('architecture') == 'whisper_encoder_v4':
        return WhisperMelFrontend(directory)
    return features


class WhisperMelFrontend:
    """Whisper log-mel calculation for a fixed two-second, 200-frame window."""
    def __init__(self, directory):
        self.filters = np.load(Path(directory) / 'whisper_mel_filters.npy', allow_pickle=False)
        if self.filters.shape not in [(80, 201), (128, 201)]: raise ValueError('Invalid Whisper mel filters')
        self.window = (.5 - .5 * np.cos(2 * np.pi * np.arange(400) / 400)).astype(np.float32)

    def __call__(self, audio):
        audio = np.asarray(audio, dtype=np.float32)
        if audio.shape != (N_SAMPLES,) or not np.isfinite(audio).all(): raise ValueError('Expected two seconds of finite audio')
        padded = np.pad(audio, (200, 200), mode='reflect')
        frames = np.lib.stride_tricks.sliding_window_view(padded, 400)[::160][:-1]
        power = np.abs(np.fft.rfft(frames * self.window, axis=-1)) ** 2
        value = np.log10(np.maximum(self.filters @ power.T, 1e-10))
        value = (np.maximum(value, value.max() - 8) + 4) / 4
        return value[None].astype(np.float32)


if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('model_directory')
    parser.add_argument('audio_wav')
    args = parser.parse_args()
    detector = WakeDetector(args.model_directory)
    print(json.dumps(detector.detect(read_wav(args.audio_wav))))
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_model.py').write_text(r'''"""Small binary CNN: no tokenizer, vocabulary, ASR model or transcript output."""
import torch
from torch import nn


class Separable(nn.Module):
    def __init__(self, source, target, stride=1):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(source, source, 3, stride=stride, padding=1, groups=source, bias=False),
                                    nn.Conv2d(source, target, 1, bias=False), nn.GroupNorm(8, target), nn.ReLU())

    def forward(self, value):
        return self.layers(value)


class WakeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 24, 5, stride=2, padding=2, bias=False),
                                     nn.GroupNorm(8, 24), nn.ReLU(), Separable(24, 48, 2),
                                     Separable(48, 64, 2), Separable(64, 96), Separable(96, 96))
        self.head = nn.Sequential(nn.Dropout(0.15), nn.Linear(96, 1))

    def forward(self, value):
        return self.head(self.encoder(value).mean(dim=(2, 3))).squeeze(-1)


class ProbabilityModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, value):
        return torch.sigmoid(self.model(value))


class Residual(nn.Module):
    def __init__(self, source, target, stride=(1, 1)):
        super().__init__()
        self.main = nn.Sequential(nn.Conv2d(source, target, 3, stride=stride, padding=1, bias=False),
                                  nn.GroupNorm(8, target), nn.SiLU(),
                                  nn.Conv2d(target, target, 3, padding=1, bias=False), nn.GroupNorm(8, target))
        self.skip = nn.Identity() if source == target and stride == (1, 1) else nn.Conv2d(source, target, 1, stride=stride, bias=False)

    def forward(self, value):
        return nn.functional.silu(self.main(value) + self.skip(value))


class TemporalWakeModel(nn.Module):
    """Retain frequency position and temporal order; attention and max pool over time."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 24, 5, stride=(2, 2), padding=2, bias=False),
                                     nn.GroupNorm(8, 24), nn.SiLU(), Residual(24, 32, (2, 1)),
                                     Residual(32, 48, (2, 1)), Residual(48, 64))
        self.project = nn.Sequential(nn.Conv1d(64 * 5, 96, 1), nn.GroupNorm(8, 96), nn.SiLU())
        self.temporal = nn.ModuleList([nn.Sequential(nn.Conv1d(96, 96, 5, dilation=d, padding=2*d, groups=96),
                                       nn.Conv1d(96, 96, 1), nn.GroupNorm(8, 96), nn.SiLU(), nn.Dropout(.1)) for d in [1, 2, 4]])
        self.attention = nn.Conv1d(96, 1, 1)
        self.head = nn.Sequential(nn.Dropout(.2), nn.Linear(192, 64), nn.SiLU(), nn.Linear(64, 1))

    def forward(self, value):
        value = self.encoder(value)
        value = self.project(value.flatten(1, 2))
        for block in self.temporal: value = value + block(value)
        weights = torch.softmax(self.attention(value), dim=-1)
        pooled = torch.cat([(value * weights).sum(-1), value.amax(-1)], dim=1)
        return self.head(pooled).squeeze(-1)


def make_model(config):
    architecture = config.get('architecture', 'legacy')
    if architecture == 'legacy': return WakeModel()
    if architecture == 'temporal_v2': return TemporalWakeModel()
    if architecture == 'speech_embedding_v3': return EmbeddingWakeModel()
    if architecture == 'whisper_encoder_v4': return WhisperWakeModel(config)
    raise ValueError(f'Unknown architecture: {architecture}')


class EmbeddingWakeModel(nn.Module):
    """Learn a keyword head on frozen speech embeddings, retaining context order."""
    def __init__(self):
        super().__init__()
        self.normalization = nn.LayerNorm(96)
        self.project = nn.Sequential(nn.Conv1d(96, 128, 1), nn.SiLU(), nn.Dropout(.15))
        self.temporal = nn.ModuleList([nn.Sequential(nn.Conv1d(128, 128, 3, padding=d, dilation=d, groups=128),
                                    nn.Conv1d(128, 128, 1), nn.GroupNorm(8, 128), nn.SiLU(), nn.Dropout(.15)) for d in [1, 2, 4]])
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(.25), nn.Linear(128 * 16, 128), nn.SiLU(), nn.Linear(128, 1))

    def forward(self, value):
        value = self.normalization(value.squeeze(1).transpose(1, 2)).transpose(1, 2)
        value = self.project(value)
        for block in self.temporal: value = value + block(value)
        return self.head(value).squeeze(-1)


class WhisperWakeModel(nn.Module):
    """Multilingual acoustic encoder plus a binary head; no decoder or text output."""
    def __init__(self, config):
        super().__init__()
        from pathlib import Path
        from transformers import WhisperConfig
        from transformers.models.whisper.modeling_whisper import WhisperEncoder
        directory = Path(config.get('encoder_dir') or Path(config['work_dir']) / 'whisper-encoder')
        encoder_config = WhisperConfig.from_json_file(str(directory / 'encoder_config.json'))
        encoder_config._attn_implementation = 'eager'
        self.encoder = WhisperEncoder(encoder_config)
        self.encoder.load_state_dict(torch.load(directory / 'encoder_state.pt', map_location='cpu', weights_only=True))
        self.encoder.requires_grad_(False)
        layers = config.get('encoder_train_layers', 2)
        if not isinstance(layers, int) or not 0 <= layers <= len(self.encoder.layers):
            raise ValueError(f'encoder_train_layers must be between 0 and {len(self.encoder.layers)}')
        if layers:
            for layer in self.encoder.layers[-layers:]: layer.requires_grad_(True)
        self.project = nn.Sequential(nn.Conv1d(encoder_config.d_model, 96, 1), nn.SiLU(), nn.Dropout(.15))
        self.attention = nn.Conv1d(96, 1, 1)
        self.head = nn.Sequential(nn.Dropout(.2), nn.Linear(192, 64), nn.SiLU(), nn.Linear(64, 1))

    def forward(self, value):
        value = self.encoder(value.squeeze(1), return_dict=False)[0].transpose(1, 2)
        value = self.project(value)
        attention = torch.softmax(self.attention(value), dim=-1)
        return self.head(torch.cat([(value * attention).sum(-1), value.amax(-1)], dim=1)).squeeze(-1)


class EnsembleLogits(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, value):
        return torch.stack([model(value) for model in self.models]).mean(0)


def spec_augment(value):
    # Small masks cannot remove the entire short wake word.
    batch, _, frequencies, frames = value.shape
    if frames == 16:
        # Speech embedding axes are latent features, not mel frequencies.
        return value + torch.randn_like(value) * .015
    time_start = torch.randint(frames - 8, (batch, 1, 1, 1), device=value.device)
    frequency_start = torch.randint(frequencies - 4, (batch, 1, 1, 1), device=value.device)
    time = torch.arange(frames, device=value.device)[None, None, None, :]
    frequency = torch.arange(frequencies, device=value.device)[None, None, :, None]
    mask = ((time >= time_start) & (time < time_start + 8)) | ((frequency >= frequency_start) & (frequency < frequency_start + 4))
    return value.masked_fill(mask, 0)
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_data.py').write_text(r'''"""Neural TTS dataset generation; separate source voices before augmentation."""
import argparse
import hashlib
import json
import math
import os
from pathlib import Path
import time
import urllib.request
import xml.sax.saxutils
import numpy as np
import torch
from scipy.signal import fftconvolve, resample_poly
from wake_audio import N_SAMPLES, SAMPLE_RATE, features, write_wav

SPLITS = ('train', 'validation', 'calibration', 'test')
RU_VOICES = {'train': ['eugene', 'kseniya'], 'validation': ['baya'], 'calibration': ['xenia'], 'test': ['aidar']}
EN_VOICES = {'train': [f'en_{i}' for i in range(12)], 'validation': [f'en_{i}' for i in range(12, 16)],
             'calibration': [f'en_{i}' for i in range(16, 20)], 'test': [f'en_{i}' for i in range(20, 24)]}
TTS = {'ru': ('v5_5_ru', 'https://models.silero.ai/models/tts/ru/v5_5_ru.pt'),
       'en': ('v3_en', 'https://models.silero.ai/models/tts/en/v3_en.pt')}
HARD_RU = ['вот', 'кот', 'год', 'рот', 'борт', 'порт', 'пот', 'тот', 'код', 'ход', 'лот', 'лёд', 'мёд',
           'болт', 'байт', 'бой', 'бок', 'быт', 'бод', 'бор', 'бота', 'боту', 'боты', 'робот', 'работа',
           'забота', 'суббота', 'ботинок', 'бутон', 'компот', 'капот', 'блок', 'бокс', 'борщ']
HARD_EN = ['boat', 'boot', 'but', 'bat', 'bit', 'bet', 'bite', 'bolt', 'box', 'boy', 'body', 'both',
           'pot', 'hot', 'dot', 'not', 'robot', 'bottle', 'bottom', 'button', 'about', 'bother']
RU_SHORT = ['да', 'нет', 'привет', 'спасибо', 'пожалуйста', 'следующий', 'пауза', 'продолжай', 'громче',
            'тише', 'стоп', 'музыка', 'радио', 'добрый день', 'доброе утро', 'добрый вечер', 'как дела',
            'почему', 'что случилось', 'подожди', 'идём домой', 'включи музыку', 'смотри сюда', 'не знаю']
EN_SHORT = ['yes', 'no', 'hello', 'thanks', 'please', 'next', 'pause', 'continue', 'louder', 'stop',
            'music', 'radio', 'good morning', 'how are you', 'wait', 'come here', 'I do not know']


def stable_seed(text):
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)


def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def download_models(work, languages):
    provenance = {}
    for language in languages:
        name, url = TTS[language]
        destination = work / 'tts' / (name + '.pt')
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            temporary = destination.with_suffix('.download')
            with urllib.request.urlopen(url, timeout=180) as response, temporary.open('wb') as output:
                import shutil
                shutil.copyfileobj(response, output)
            temporary.replace(destination)
        provenance[language] = {'name': name, 'url': url, 'sha256': hashlib.sha256(destination.read_bytes()).hexdigest(),
                                'license_reference': 'https://github.com/snakers4/silero-models#licence'}
    atomic_json(work / 'tts_provenance.json', provenance)


def negative_texts(language, target, standalone, exclusions=()):
    if language == 'ru':
        phrases = [f'{verb} {noun}' for verb in ['включи', 'выключи', 'покажи', 'найди', 'добавь', 'убери', 'проверь', 'запусти']
                   for noun in ['песню', 'радио', 'музыку', 'трек', 'альбом', 'плеер', 'очередь', 'запись']]
        context = [f'этот {target}', f'наш {target}', f'где {target}', f'{target} работает', f'{target} молчит',
                   f'это {target}', f'{target} готов', f'новый {target}'] if standalone else []
        words = HARD_RU + RU_SHORT
    else:
        phrases = [f'{verb} {noun}' for verb in ['play', 'stop', 'find', 'show', 'start', 'check', 'add', 'remove']
                   for noun in ['music', 'radio', 'song', 'track', 'album', 'player', 'queue', 'audio']]
        context = [f'this {target}', f'our {target}', f'the {target}', f'where is {target}', f'{target} works',
                   f'new {target}', f'{target} is ready', f'hello {target}'] if standalone else []
        words = HARD_EN + EN_SHORT
    target = target.strip().lower()
    # Changing the target cannot turn that exact word into a negative example.
    words = [text for text in words if target not in text.lower().split()]
    phrases = [text for text in phrases if target not in text.lower().split()]
    excluded = {text.strip().casefold() for text in exclusions}
    return sorted(text for text in set(words + phrases + context) if text.strip().casefold() not in excluded)


def background(rng, length):
    kind = int(rng.integers(5))
    if kind == 0:
        return np.zeros(length, np.float32)
    white = rng.normal(0, 1, length)
    if kind == 1:
        sound = white
    elif kind == 2:
        sound = np.cumsum(white)
        sound -= sound.mean()
    elif kind == 3:
        t = np.arange(length) / SAMPLE_RATE
        sound = sum(np.sin(2 * np.pi * rng.uniform(70, 1200) * t + rng.uniform(0, 6.28)) for _ in range(4))
        sound *= 0.5 + 0.5 * np.sin(2 * np.pi * rng.uniform(1, 5) * t)
    else:
        sound = white * 0.05
        for _ in range(10):
            index = int(rng.integers(length - 100))
            sound[index:index + 100] += rng.uniform(-10, 10) * np.exp(-np.arange(100) / 20)
    return (sound / (np.sqrt(np.mean(sound ** 2)) + 1e-7)).astype(np.float32)


def prepare_source(audio):
    if not len(audio) or not np.isfinite(audio).all() or np.max(np.abs(audio)) < 1e-4:
        raise ValueError('TTS produced empty/silent audio')
    active = np.flatnonzero(np.abs(audio) > max(0.001, np.max(np.abs(audio)) * 0.02))
    if not len(active):
        raise ValueError('No audible speech in TTS output')
    return audio[max(0, active[0] - 800):min(len(audio), active[-1] + 801)]


def augment(audio, rng, label, preserve_context=False):
    # Resampling changes pitch and tempo together; no claim of independent voice generation.
    ratio = int(rng.integers(85, 116))
    speech = resample_poly(audio, ratio, 100).astype(np.float32)
    if len(speech) > N_SAMPLES - 800:
        if label or preserve_context:
            return None  # Never crop away the wake or context while retaining its label.
        start = int(rng.integers(len(speech) - (N_SAMPLES - 800) + 1))
        speech = speech[start:start + N_SAMPLES - 800]
    if rng.random() < 0.5:
        ir_length = int(rng.integers(800, 4000))
        ir = rng.normal(0, 0.03, ir_length) * np.exp(-np.arange(ir_length) / rng.uniform(200, 1500))
        ir[0] = 1
        speech = fftconvolve(speech, ir)[:len(speech)].astype(np.float32)
    speech *= 10 ** (rng.uniform(-24, -3) / 20) / (np.max(np.abs(speech)) + 1e-6)
    window = np.zeros(N_SAMPLES, np.float32)
    start = int(rng.integers(N_SAMPLES - len(speech) + 1))
    window[start:start + len(speech)] = speech
    noise = background(rng, N_SAMPLES)
    if np.any(noise):
        level = max(float(np.sqrt(np.mean(speech ** 2))), 1e-4) / 10 ** (rng.uniform(5, 30) / 20)
        window += noise * level
    if rng.random() < 0.2:
        # Bandwidth loss approximates a low-quality microphone; not a Discord Opus simulation.
        window = resample_poly(resample_poly(window, 1, 2), 2, 1)[:N_SAMPLES]
    peak = np.max(np.abs(window))
    if peak > 0.98:
        window *= 0.98 / peak
    return window.astype(np.float32)


def voice_jobs(config):
    jobs = []
    for language, splits in [('ru', RU_VOICES), ('en', EN_VOICES)]:
        if language == 'en' and not config['use_english']:
            continue
        for split, voices in splits.items():
            for voice in voices:
                jobs.append({'language': language, 'voice': voice, 'split': split})
    return jobs


def synthesize(model, text, speaker, device, rate=None):
    if rate:
        escaped = xml.sax.saxutils.escape(text)
        kwargs = {'ssml_text': f'<speak><prosody rate="{rate}">{escaped}</prosody></speak>'}
    else:
        kwargs = {'text': text}
    with torch.inference_mode():
        audio = model.apply_tts(**kwargs, speaker=speaker, sample_rate=24000)
    audio = audio.detach().cpu().float().numpy()
    return prepare_source(resample_poly(audio, 2, 3).astype(np.float32))


def generate(config, rank=0, world=1):
    work = Path(config['work_dir'])
    torch.set_num_threads(config.get('cpu_threads', 2))
    device = f'cuda:{rank}' if torch.cuda.is_available() and not config.get('force_cpu') else 'cpu'
    if device != 'cpu':
        torch.cuda.set_device(rank)
    jobs = voice_jobs(config)[rank::world]
    smoke = config.get('smoke', False)
    destination = work / 'dataset'
    for folder in ['wav', 'features', 'sources']:
        (destination / folder).mkdir(parents=True, exist_ok=True)
    rows = []
    model, current_language = None, None
    started = time.monotonic()
    for job in jobs:
        language, speaker, split = job['language'], job['voice'], job['split']
        name = TTS[language][0]
        if language != current_language:
            if model is not None:
                del model
                if device != 'cpu': torch.cuda.empty_cache()
            model = torch.package.PackageImporter(str(work / 'tts' / (name + '.pt'))).load_pickle('tts_models', 'model')
            model.to(torch.device(device))
            current_language = language
        target = config['wake_word_ru' if language == 'ru' else 'wake_word_en'].strip()
        positive_texts = [target + mark for mark in ['.', '!', '?']]
        if not config['standalone_only']:
            positive_texts += [f'{target}, привет.' if language == 'ru' else f'{target}, hello.']
        negative_text = negative_texts(language, target, config['standalone_only'])
        sources = {0: [], 1: []}
        for label, texts in [(1, positive_texts), (0, negative_text[:6] if smoke else negative_text)]:
            rates = [None, 'slow', 'fast'] if label else [None]
            for text in texts:
                for rate in rates:
                    source_id = hashlib.sha256(f'{name}|{speaker}|{text}|{rate}'.encode()).hexdigest()[:24]
                    file = destination / 'sources' / (source_id + '.wav')
                    if file.exists():
                        from wake_audio import read_wav
                        audio = read_wav(file)
                    else:
                        audio = synthesize(model, text, speaker, device, rate)
                        write_wav(file, audio)
                        from wake_audio import read_wav
                        audio = read_wav(file)  # Consistent PCM16 quantization on rerun.
                    preserve = label == 1 or target.lower() in text.lower().replace(',', '').split()
                    sources[label].append((source_id, text, audio, preserve))
        for label in [1, 0]:
            count = config['positive_per_voice'] if label else config['negative_per_voice']
            rng = np.random.default_rng(config['seed'] + stable_seed(f'{language}:{speaker}:{label}'))
            attempts, written = 0, 0
            while written < count:
                attempts += 1
                if attempts > count * 30:
                    raise RuntimeError('Too many overlong wake/context examples; use a shorter word')
                source_id, text, audio, preserve = sources[label][int(rng.integers(len(sources[label])))]
                window = augment(audio, rng, label, preserve)
                if window is None: continue
                identifier = f'{language}-{speaker}-{label}-{written:05d}'
                wav = destination / 'wav' / (identifier + '.wav')
                write_wav(wav, window)
                from wake_audio import read_wav
                feat = features(read_wav(wav))
                np.save(destination / 'features' / (identifier + '.npy'), feat)
                rows.append({'id': identifier, 'label': label, 'split': split, 'language': language,
                             'speaker': f'{name}/{speaker}', 'source_id': source_id, 'text': text,
                             'wav': f'wav/{identifier}.wav', 'feature': f'features/{identifier}.npy', 'generator': 'neural_tts'})
                written += 1
        print(f'[TTS GPU {rank}] {language}/{speaker} -> {split}: {count + config["positive_per_voice"]} windows', flush=True)
    # Every noise seed belongs to one split; both labels receive the same noise process.
    for index in range(rank, config['noise_windows'], world):
        split = SPLITS[index % 4]
        rng = np.random.default_rng(config['seed'] + 100000 + index)
        audio = background(rng, N_SAMPLES) * 10 ** (rng.uniform(-45, -12) / 20)
        audio = np.clip(audio, -0.98, 0.98)
        identifier = f'noise-{index:05d}'
        write_wav(destination / 'wav' / (identifier + '.wav'), audio)
        from wake_audio import read_wav
        np.save(destination / 'features' / (identifier + '.npy'), features(read_wav(destination / 'wav' / (identifier + '.wav'))))
        rows.append({'id': identifier, 'label': 0, 'split': split, 'language': 'none', 'speaker': 'noise',
                     'source_id': identifier, 'text': '', 'wav': f'wav/{identifier}.wav',
                     'feature': f'features/{identifier}.npy', 'generator': 'procedural_noise'})
    shard = destination / f'manifest-rank-{rank}.jsonl'
    temporary = shard.with_suffix('.tmp')
    temporary.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')
    temporary.replace(shard)
    print(f'[TTS rank {rank}] finished {len(rows)} windows in {time.monotonic() - started:.1f}s on {device}', flush=True)


def merge_manifest(config, world):
    work = Path(config['work_dir'])
    rows = []
    for rank in range(world):
        file = work / 'dataset' / f'manifest-rank-{rank}.jsonl'
        rows += [json.loads(line) for line in file.read_text(encoding='utf-8').splitlines() if line]
    identifiers, group_split, speakers = set(), {}, {}
    for row in rows:
        assert row['id'] not in identifiers, 'Duplicate generated ID'
        identifiers.add(row['id'])
        assert group_split.setdefault(row['source_id'], row['split']) == row['split'], 'Source audio leakage'
        if row['speaker'] != 'noise':
            assert speakers.setdefault(row['speaker'], row['split']) == row['split'], 'Speaker leakage'
        feature = np.load(work / 'dataset' / row['feature'])
        expected_shape = {'speech_embedding_v3': (1, 96, 16), 'whisper_encoder_v4': (1, 80, 200)}.get(config.get('architecture'), (1, 40, 201))
        assert feature.shape == expected_shape and np.isfinite(feature).all()
    for split in SPLITS:
        assert {row['label'] for row in rows if row['split'] == split} == {0, 1}
    rows.sort(key=lambda row: row['id'])
    (work / 'dataset' / 'manifest.jsonl').write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')
    summary = {split: {'positive': sum(r['split'] == split and r['label'] == 1 for r in rows),
                       'negative': sum(r['split'] == split and r['label'] == 0 for r in rows),
                       'voices': len({r['speaker'] for r in rows if r['split'] == split and r['speaker'] != 'noise'})} for split in SPLITS}
    atomic_json(work / 'dataset_summary.json', summary)
    return rows, summary


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--rank', type=int, default=0)
    parser.add_argument('--world', type=int, default=1)
    args = parser.parse_args()
    generate(json.loads(Path(args.config).read_text(encoding='utf-8')), args.rank, args.world)
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_tts.py').write_text(r'''"""Four local neural TTS families. Voice identities stay in one split across versions."""
import hashlib
import json
import math
from pathlib import Path
import urllib.request
import xml.sax.saxutils
import numpy as np
import torch
from scipy.signal import resample_poly

SILERO_RU = {'train': ['eugene', 'kseniya'], 'validation': ['baya'], 'calibration': ['xenia'], 'test': ['aidar']}
SILERO_EN = {'train': [f'en_{i}' for i in range(16)], 'validation': ['en_16', 'en_17'],
             'calibration': ['en_18', 'en_19'], 'test': [f'en_{i}' for i in range(20, 24)]}
PIPER_RU = {'train': ['ru_RU-dmitri-medium', 'ru_RU-ruslan-medium', 'ru_RU-irina-medium'], 'validation': [],
            'calibration': [], 'test': ['ru_RU-denis-medium']}
PIPER_EN = {'train': ['en_US-amy-medium', 'en_US-lessac-medium', 'en_US-ryan-medium'],
            'validation': ['en_US-hfc_female-medium'], 'calibration': ['en_US-joe-medium'],
            'test': ['en_US-hfc_male-medium']}
KOKORO_EN = {'train': ['af_heart', 'af_bella', 'af_nicole', 'af_sarah', 'af_aoede', 'af_kore', 'af_nova',
                       'am_fenrir', 'am_michael', 'am_puck', 'am_echo', 'am_onyx', 'bf_emma', 'bm_fable'],
             'validation': ['af_alloy', 'bm_george'], 'calibration': ['af_sky', 'bf_isabella'],
             'test': ['af_jessica', 'bm_lewis']}
SILERO_MODELS = {'v5_5_ru': 'https://models.silero.ai/models/tts/ru/v5_5_ru.pt',
                 'v4_ru': 'https://models.silero.ai/models/tts/ru/v4_ru.pt',
                 'v3_en': 'https://models.silero.ai/models/tts/en/v3_en.pt'}
KOKORO_BASE = 'https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.1/'
PIPER_BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/'


def file_sha256(file):
    digest = hashlib.sha256()
    with Path(file).open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def voice_jobs(config):
    jobs = []
    def add(engine, model, language, voices):
        if language == 'en' and not config['use_english']: return
        if engine not in config.get('tts_engines', ['silero', 'piper', 'mms', 'kokoro']): return
        for split, speakers in voices.items():
            for speaker in speakers:
                # Different Silero versions are renderings of the same voice, not new speakers.
                jobs.append(dict(engine=engine, model=model, language=language, voice=speaker, split=split,
                                 speaker_group=f'{engine}/{language}/{speaker}'))
    silero_ru, piper_ru = SILERO_RU, PIPER_RU
    if config.get('gender_balanced_ru') is False:
        silero_ru = dict(train=['eugene', 'kseniya', 'baya'], validation=[], calibration=['xenia'], test=['aidar'])
        piper_ru = dict(train=['ru_RU-dmitri-medium', 'ru_RU-ruslan-medium'], validation=['ru_RU-irina-medium'], calibration=[], test=['ru_RU-denis-medium'])
    if config.get('piper_ru_splits') is not None:
        proposed = config['piper_ru_splits']
        known = {voice for voices in PIPER_RU.values() for voice in voices}
        if set(proposed) != {'train', 'validation', 'calibration', 'test'}:
            raise ValueError('Piper RU override requires all four splits')
        seen = set()
        for split, voices in proposed.items():
            if not isinstance(voices, list) or not voices:
                raise ValueError('Each Piper RU split requires at least one voice')
            for voice in voices:
                if voice not in known or voice in seen:
                    raise ValueError('Piper RU override contains an unknown or repeated voice')
                seen.add(voice)
        piper_ru = proposed
    for model in config.get('silero_ru_versions', ['v5_5_ru', 'v4_ru']): add('silero', model, 'ru', silero_ru)
    add('silero', 'v3_en', 'en', SILERO_EN)
    add('piper', 'piper', 'ru', piper_ru)
    add('piper', 'piper', 'en', PIPER_EN)
    add('kokoro', 'kokoro-v1.0', 'en', KOKORO_EN)
    add('mms', 'facebook/mms-tts-rus', 'ru', {'train': ['rus']})
    add('mms', 'facebook/mms-tts-eng', 'en', {'train': ['eng']})
    return jobs


def piper_relative(voice):
    locale, speaker, quality = voice.split('-')
    return f'{locale[:2]}/{locale}/{speaker}/{quality}/{voice}.onnx'


def download(url, file):
    file = Path(file)
    file.parent.mkdir(parents=True, exist_ok=True)
    if not file.exists():
        temporary = file.with_suffix(file.suffix + '.download')
        with urllib.request.urlopen(url, timeout=240) as response, temporary.open('wb') as output:
            import shutil
            shutil.copyfileobj(response, output)
        temporary.replace(file)
    return dict(url=url, sha256=hashlib.sha256(file.read_bytes()).hexdigest(), bytes=file.stat().st_size)


def prepare_embedding_models(work):
    from wake_data import atomic_json
    directory = Path(work) / 'speech-embedding'
    provenance = {name: download('https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/' + name,
                                directory / name) for name in ['melspectrogram.onnx', 'embedding_model.onnx']}
    expected = {'melspectrogram.onnx': 'ba2b0e0f8b7b875369a2c89cb13360ff53bac436f2895cced9f479fa65eb176f',
                'embedding_model.onnx': '70d164290c1d095d1d4ee149bc5e00543250a7316b59f31d056cff7bd3075c1f'}
    for name, checksum in expected.items():
        if provenance[name]['sha256'] != checksum:
            raise ValueError(f'Pretrained frontend weights changed: {name}')
    provenance['reference'] = 'https://github.com/dscripka/openWakeWord#model-architecture'
    provenance['licence_note'] = 'Upstream describes Google speech_embedding as Apache-2.0; distributed pre-trained models have CC-BY-NC-SA terms. Keep upstream licence attribution; no commercial licence guarantee.'
    atomic_json(directory / 'provenance.json', provenance)
    return provenance


def prepare_whisper_encoder(work, repository='openai/whisper-tiny', revision=None):
    from huggingface_hub import snapshot_download, HfApi
    from transformers import WhisperConfig, WhisperFeatureExtractor
    from transformers.models.whisper.modeling_whisper import WhisperEncoder
    from safetensors import safe_open
    from wake_data import atomic_json
    directory = Path(work) / 'whisper-encoder'
    directory.mkdir(parents=True, exist_ok=True)
    provenance_file = directory / 'provenance.json'
    previous = json.loads(provenance_file.read_text(encoding='utf-8')) if provenance_file.exists() else None
    if previous and previous['repository'] != repository:
        raise ValueError('Encoder directory belongs to another model')
    revision = revision or (previous['revision'] if previous else (
        '169d4a4341b33bc18d8881c4b69c2e104e1cc0af' if repository == 'openai/whisper-tiny' else HfApi().model_info(repository).sha))
    if previous and previous['revision'] != revision:
        raise ValueError('Encoder revision changed; use a new directory')
    if not all((directory / name).exists() for name in ['encoder_config.json', 'encoder_state.pt', 'whisper_mel_filters.npy']):
        source = directory / 'source'
        snapshot_download(repo_id=repository, revision=revision, local_dir=source,
                          allow_patterns=['config.json', 'preprocessor_config.json', 'model.safetensors'])
        encoder_config = WhisperConfig.from_pretrained(source, local_files_only=True)
        encoder_config._attn_implementation = 'eager'
        encoder = WhisperEncoder(encoder_config)
        # Read only encoder tensors; never instantiate the much larger decoder.
        with safe_open(str(source / 'model.safetensors'), framework='pt', device='cpu') as saved:
            state = {key.removeprefix('model.encoder.'): saved.get_tensor(key)
                     for key in saved.keys() if key.startswith('model.encoder.')}
        if not state:
            raise ValueError('No encoder tensors in Whisper checkpoint')
        encoder.load_state_dict(state)
        del state
        encoder.config.max_source_positions = 100
        encoder.max_source_positions = 100
        encoder.embed_positions = torch.nn.Embedding.from_pretrained(encoder.embed_positions.weight[:100].detach().clone(), freeze=True)
        encoder.config.to_json_file(str(directory / 'encoder_config.json'))
        torch.save(encoder.state_dict(), directory / 'encoder_state.pt')
        extractor = WhisperFeatureExtractor.from_pretrained(source, local_files_only=True)
        np.save(directory / 'whisper_mel_filters.npy', extractor.mel_filters.T.astype(np.float32))
    provenance = dict(repository=repository, revision=revision, window_adaptation='encoder positions truncated to 100 for 2-second audio',
                      license_reference=f'https://huggingface.co/{repository}',
                      files_sha256={name: file_sha256(directory / name)
                                    for name in ['encoder_config.json', 'encoder_state.pt', 'whisper_mel_filters.npy']})
    atomic_json(directory / 'provenance.json', provenance)
    return provenance


def download_models(work, config):
    if config.get('architecture') == 'speech_embedding_v3':
        prepare_embedding_models(work)
    if config.get('architecture') == 'whisper_encoder_v4':
        prepare_whisper_encoder(work)
    work = Path(work)
    provenance = {}
    jobs = voice_jobs(config)
    for model in sorted({j['model'] for j in jobs if j['engine'] == 'silero'}):
        provenance[model] = download(SILERO_MODELS[model], work / 'tts' / (model + '.pt'))
        provenance[model]['license_reference'] = 'https://github.com/snakers4/silero-models#licence'
    for voice in sorted({j['voice'] for j in jobs if j['engine'] == 'piper'}):
        relative = piper_relative(voice)
        provenance[voice] = download(PIPER_BASE + relative, work / 'tts/piper' / (voice + '.onnx'))
        provenance[voice]['config'] = download(PIPER_BASE + relative + '.json', work / 'tts/piper' / (voice + '.onnx.json'))
        card = relative.rsplit('/', 1)[0] + '/MODEL_CARD'
        provenance[voice]['license_reference'] = PIPER_BASE + card
    if any(j['engine'] == 'kokoro' for j in jobs):
        for file in ['kokoro-v1.0.onnx', 'voices-v1.0.bin']:
            provenance[file] = download(KOKORO_BASE + file, work / 'tts/kokoro' / file)
            provenance[file]['license_reference'] = 'https://huggingface.co/hexgrad/Kokoro-82M'
    if any(j['engine'] == 'mms' for j in jobs):
        from huggingface_hub import snapshot_download
        for repo in sorted({j['model'] for j in jobs if j['engine'] == 'mms'}):
            local = work / 'tts/mms' / repo.split('/')[-1]
            snapshot_download(repo_id=repo, local_dir=local,
                              allow_patterns=['*.json', '*.safetensors', 'pytorch_model.bin', '*.txt'])
            hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in local.iterdir() if p.is_file()}
            provenance[repo] = dict(repository=repo, files_sha256=hashes,
                                    license_reference=f'https://huggingface.co/{repo}')
    file = work / 'tts_provenance.json'
    file.write_text(json.dumps(provenance, indent=2), encoding='utf-8')
    return provenance


def resample(audio, sample_rate):
    common = math.gcd(16000, int(sample_rate))
    return resample_poly(np.asarray(audio, dtype=np.float32).reshape(-1), 16000 // common,
                         int(sample_rate) // common).astype(np.float32)


class Synthesizer:
    def __init__(self, work, job, device, threads=2):
        self.work, self.job, self.device = Path(work), job, device
        self.engine = job['engine']
        if self.engine == 'silero':
            self.model = torch.package.PackageImporter(str(self.work / 'tts' / (job['model'] + '.pt'))).load_pickle('tts_models', 'model')
            self.model.to(torch.device(device))
            self.provider = device
        elif self.engine == 'piper':
            import onnxruntime as ort
            from piper import PiperVoice
            from piper.config import PiperConfig
            file = self.work / 'tts/piper' / (job['voice'] + '.onnx')
            options = ort.SessionOptions(); options.intra_op_num_threads = threads; options.inter_op_num_threads = 1
            session = ort.InferenceSession(str(file), options, providers=['CPUExecutionProvider'])
            self.model = PiperVoice(session=session, config=PiperConfig.from_dict(json.loads(Path(str(file) + '.json').read_text(encoding='utf-8'))))
            self.provider = 'CPUExecutionProvider'
        elif self.engine == 'kokoro':
            import onnxruntime as ort
            from kokoro_onnx import Kokoro
            options = ort.SessionOptions()
            options.intra_op_num_threads = threads
            options.inter_op_num_threads = 1
            session = ort.InferenceSession(str(self.work / 'tts/kokoro/kokoro-v1.0.onnx'), sess_options=options,
                                           providers=['CPUExecutionProvider'])
            self.model = Kokoro.from_session(session, str(self.work / 'tts/kokoro/voices-v1.0.bin'))
            self.provider = 'CPUExecutionProvider'
        elif self.engine == 'mms':
            from transformers import VitsModel, VitsTokenizer
            local = self.work / 'tts/mms' / job['model'].split('/')[-1]
            self.tokenizer = VitsTokenizer.from_pretrained(local)
            self.model = VitsModel.from_pretrained(local).to(device).eval()
            self.provider = device
        else: raise ValueError(f'Unknown TTS engine: {self.engine}')

    def say(self, text, rate='normal', seed=0):
        speed = {'normal': 1., 'slow': .85, 'fast': 1.15}[rate]
        torch.manual_seed(seed)
        np.random.seed(seed % (2 ** 32))
        if self.engine == 'silero':
            args = {'text': text} if rate == 'normal' else {'ssml_text':
                f'<speak><prosody rate="{rate}">{xml.sax.saxutils.escape(text)}</prosody></speak>'}
            with torch.inference_mode():
                audio = self.model.apply_tts(**args, speaker=self.job['voice'], sample_rate=24000).cpu().numpy()
            sample_rate = 24000
        elif self.engine == 'piper':
            from piper import SynthesisConfig
            chunks = list(self.model.synthesize(text, SynthesisConfig(length_scale=1 / speed, noise_scale=.667, noise_w_scale=.8)))
            if not chunks: raise ValueError('Empty Piper result')
            audio = np.concatenate([c.audio_float_array for c in chunks])
            sample_rate = chunks[0].sample_rate
        elif self.engine == 'kokoro':
            # 0.4.9 assumes int32 speed for input_ids exports; the released graph
            # uses float32. Build inputs from actual graph metadata instead.
            language = 'en-gb' if self.job['voice'].startswith('b') else 'en-us'
            phonemes = self.model.tokenizer.phonemize(text, language)
            tokens = self.model.tokenizer.tokenize(phonemes)
            if not tokens or len(tokens) > 510: raise ValueError('Kokoro text outside complete short-utterance limit')
            style = self.model.get_voice_style(self.job['voice'])
            style = style[min(len(tokens), len(style)) - 1]
            values = {'tokens': [[0, *tokens, 0]], 'input_ids': [[0, *tokens, 0]], 'style': style, 'speed': [speed]}
            dtypes = {'tensor(float)': np.float32, 'tensor(int64)': np.int64, 'tensor(int32)': np.int32}
            inputs = {item.name: np.asarray(values[item.name], dtype=dtypes[item.type]) for item in self.model.sess.get_inputs()}
            audio = self.model.sess.run(None, inputs)[0]
            sample_rate = 24000
        else:
            inputs = self.tokenizer(text, return_tensors='pt').to(self.device)
            old = self.model.speaking_rate
            self.model.speaking_rate = speed
            try:
                with torch.inference_mode(): audio = self.model(**inputs).waveform.cpu().numpy().reshape(-1)
            finally: self.model.speaking_rate = old
            sample_rate = self.model.config.sampling_rate
        return resample(audio, sample_rate)
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_data_v2.py').write_text(r'''"""Speaker-held-out multi-engine dataset with deliberate hard negatives and real speech."""
import argparse
import hashlib
import heapq
import json
import re
from pathlib import Path
import tarfile
import time
import zipfile
import numpy as np
import torch
from scipy.signal import fftconvolve, resample_poly
from wake_audio import SAMPLE_RATE, N_SAMPLES, read_wav, write_wav, right_aligned_window
from wake_data import atomic_json, background, negative_texts, prepare_source, stable_seed, SPLITS
from wake_tts import Synthesizer, voice_jobs, download_models, download

REAL_URLS = {'mini': 'https://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip',
             'full': 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'}
REAL_WORDS = set('backward bed bird cat dog down eight five follow forward four go happy house learn left marvin nine no off on one right seven sheila six stop three tree two up visual wow yes zero'.split())


def speaker_split(speaker):
    bucket = stable_seed('real-speaker:' + speaker) % 100
    return 'train' if bucket < 76 else 'validation' if bucket < 84 else 'calibration' if bucket < 92 else 'test'


def prepare_real(config):
    work = Path(config['work_dir'])
    count = config.get('real_speech_windows', 12000)
    if not count:
        atomic_json(work / 'real_sources.json', [])
        return
    source = config.get('real_speech_source', 'full')
    archive = work / 'downloads' / ('mini.zip' if source == 'mini' else 'speech-commands-v002.tar.gz')
    provenance = download(REAL_URLS[source], archive)
    provenance['license_reference'] = 'https://www.tensorflow.org/datasets/catalog/speech_commands'
    provenance['description'] = 'Known-word negatives only; speaker assigned before augmentation; no real wake positives.'
    target = {config['wake_word_ru'].lower(), config['wake_word_en'].lower()}
    destination = work / 'dataset/real-sources'
    destination.mkdir(parents=True, exist_ok=True)
    budgets = dict(train=int(count * .76), validation=int(count * .08), calibration=int(count * .08))
    budgets['test'] = count - sum(budgets.values())
    reservoirs = {split: [] for split in SPLITS}
    backgrounds = []
    def accept(name, data):
        parts = name.replace('\\', '/').split('/')
        if len(parts) < 2: return
        word, filename = parts[-2:]
        if not re.fullmatch(r'[A-Za-z0-9_]+\.wav', filename): return
        if word == '_background_noise_':
            path = destination / ('background-' + filename)
            path.write_bytes(data)
            backgrounds.append(str(path))
            return
        if word not in REAL_WORDS or word in target or '_nohash_' not in filename: return
        speaker = filename.split('_nohash_')[0]
        identifier = 'real-' + hashlib.sha256((word + '/' + filename).encode()).hexdigest()[:24]
        split = speaker_split(speaker)
        priority = stable_seed(identifier + str(config['seed']))
        entry = (-priority, identifier, dict(id=identifier, word=word, speaker='speechcommands/' + speaker,
                           speaker_group='speechcommands/' + speaker, split=split, data=data))
        reservoir, budget = reservoirs[split], budgets[split]
        if not budget: return
        if len(reservoir) < budget: heapq.heappush(reservoir, entry)
        elif priority < -reservoir[0][0]: heapq.heapreplace(reservoir, entry)
    # Read member bytes only; never extract archive paths or links to the filesystem.
    if source == 'mini':
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                if member.filename.endswith('.wav') and member.file_size <= 8_000_000:
                    accept(member.filename, zipped.read(member))
    else:
        with tarfile.open(archive, 'r|gz') as tar:
            for member in tar:
                if member.isfile() and member.name.endswith('.wav') and member.size <= 8_000_000:
                    accept(member.name, tar.extractfile(member).read())
    selected = []
    for split in SPLITS:
        group = [item[2] for item in reservoirs[split]]
        group.sort(key=lambda r: stable_seed(r['id'] + str(config['seed'])))
        for row in group[:budgets[split]]:
            payload = row.pop('data')
            file = destination / (row['id'] + '.wav')
            file.write_bytes(payload)
            row['file'] = str(file)
            selected.append(row)
    atomic_json(work / 'real_sources.json', selected)
    atomic_json(work / 'real_backgrounds.json', backgrounds)
    provenance['selected_windows'] = len(selected)
    provenance['corpus'] = source
    atomic_json(work / 'real_provenance.json', provenance)
    print(f'Real speech negatives: {len(selected)}; background recordings: {len(backgrounds)}', flush=True)


def negative_kind(text, target):
    tokens = re.findall(r"[\w]+", text.lower())
    if target.lower() in tokens: return 'context'
    return 'hard_word' if len(tokens) == 1 else 'ordinary'


def prepare_human(config):
    """Optional reviewed recordings: labels come from annotation, not an ASR teacher."""
    if not config.get('real_wake_manifest'): return
    work = Path(config['work_dir'])
    manifest = Path(config['real_wake_manifest'])
    records = [r for r in json.loads((work / 'real_sources.json').read_text(encoding='utf-8'))
               if r.get('generator') != 'human_reviewed']
    groups, audio_groups = {}, {}
    destination = work / 'dataset/real-sources'
    destination.mkdir(parents=True, exist_ok=True)
    for line in manifest.read_text(encoding='utf-8').splitlines():
        if not line.strip(): continue
        row = json.loads(line)
        if row.get('reviewed') is not True or type(row.get('label')) is not int or row['label'] not in [0, 1]:
            raise ValueError('Human recordings require reviewed=true and an explicit binary label')
        if row.get('split') not in SPLITS or row.get('language') not in ['ru', 'en'] or not row.get('speaker'):
            raise ValueError('Human recordings require speaker, language and split')
        group = 'human/' + str(row['speaker'])
        if groups.setdefault(group, row['split']) != row['split']: raise ValueError('Human speaker leakage')
        file = Path(row['wav'])
        if not file.is_absolute(): file = manifest.parent / file
        audio = prepare_source(read_wav(file))
        if len(audio) > N_SAMPLES - 480: raise ValueError('Human annotations must contain a complete utterance shorter than two seconds')
        sha = hashlib.sha256(audio.tobytes()).hexdigest()
        signature = (row['split'], row['label'])
        if audio_groups.setdefault(sha, signature) != signature: raise ValueError('Human audio leakage or contradictory labels')
        identifier = 'human-' + sha[:24]
        if any(r['id'] == identifier for r in records): continue
        saved = destination / (identifier + '.wav')
        write_wav(saved, audio)
        records.append(dict(id=identifier, word=row.get('text', ''), speaker=group, speaker_group=group,
                            split=row['split'], label=row['label'], language=row['language'], generator='human_reviewed', file=str(saved)))
    atomic_json(work / 'real_sources.json', records)


def augment(audio, rng, label, preserve_context=False, *, train=False, noise_pool=None):
    ratio = int(rng.integers(93, 108))
    speech = resample_poly(audio, ratio, 100).astype(np.float32)
    if len(speech) > N_SAMPLES - 480:
        if label or preserve_context: return None
        start = int(rng.integers(len(speech) - (N_SAMPLES - 480) + 1))
        speech = speech[start:start + N_SAMPLES - 480]
    if rng.random() < .3:
        ir = rng.normal(0, .025, int(rng.integers(400, 2400)))
        ir *= np.exp(-np.arange(len(ir)) / rng.uniform(160, 850))
        ir[0] = 1
        speech = fftconvolve(speech, ir)[:len(speech)].astype(np.float32)
    speech *= 10 ** (rng.uniform(-22, -3) / 20) / (np.max(np.abs(speech)) + 1e-6)
    window = np.zeros(N_SAMPLES, np.float32)
    # Match exported inference's right-aligned utterance; retain some position variation.
    start = N_SAMPLES - len(speech) if rng.random() < .65 else int(rng.integers(N_SAMPLES - len(speech) + 1))
    window[start:start + len(speech)] = speech
    if rng.random() < .65:
        if train and noise_pool and rng.random() < .6:
            noise = noise_pool[int(rng.integers(len(noise_pool)))]
            offset = int(rng.integers(max(1, len(noise) - N_SAMPLES + 1)))
            noise = np.resize(noise[offset:offset + N_SAMPLES], N_SAMPLES).copy()
            noise /= max(float(np.sqrt(np.mean(noise ** 2))), 1e-6)
        else: noise = background(rng, N_SAMPLES)
        snr = rng.uniform(8, 35)  # Most samples remain audible; no wholesale low-SNR label corruption.
        window += noise * max(float(np.sqrt(np.mean(speech ** 2))), 1e-4) / 10 ** (snr / 20)
    if rng.random() < .15:
        window = resample_poly(resample_poly(window, 1, 2), 2, 1)[:N_SAMPLES]
    if train and rng.random() < .12:
        # Mild saturation/quantization; does not claim exact Opus packet simulation.
        window = np.tanh(window * rng.uniform(1, 2))
        window = np.round(window * 2047) / 2047
    peak = np.max(np.abs(window))
    if peak > .98: window *= .98 / peak
    return window.astype(np.float32)


def generate(config, rank=0, world=1):
    work = Path(config['work_dir'])
    torch.set_num_threads(config.get('cpu_threads', 2))
    device = f'cuda:{rank}' if torch.cuda.is_available() and not config.get('force_cpu') else 'cpu'
    if device != 'cpu': torch.cuda.set_device(rank)
    destination = work / 'dataset'
    for folder in ['wav', 'features', 'sources']: (destination / folder).mkdir(parents=True, exist_ok=True)
    real = json.loads((work / 'real_sources.json').read_text(encoding='utf-8')) if (work / 'real_sources.json').exists() else []
    backgrounds = json.loads((work / 'real_backgrounds.json').read_text(encoding='utf-8')) if (work / 'real_backgrounds.json').exists() else []
    noise_pool = [read_wav(p) for p in backgrounds]
    # Training-only real background speech. Held-out speakers never enter training mixtures.
    noise_pool += [read_wav(r['file']) for r in real if r['split'] == 'train' and r.get('generator', 'real_speech') == 'real_speech'][:64]
    rows, audit = [], []
    started = time.monotonic()
    synthesizer, key = None, None
    from wake_audio import make_frontend
    frontend = make_frontend(config, work / ('whisper-encoder' if config.get('architecture') == 'whisper_encoder_v4' else 'speech-embedding'))
    def save(identifier, audio, row):
        write_wav(destination / 'wav' / (identifier + '.wav'), audio)
        np.save(destination / 'features' / (identifier + '.npy'), frontend(read_wav(destination / 'wav' / (identifier + '.wav'))))
        rows.append(dict(id=identifier, wav=f'wav/{identifier}.wav', feature=f'features/{identifier}.npy', **row))
    for job in voice_jobs(config)[rank::world]:
        engine, model, language, speaker, split = [job[k] for k in ['engine', 'model', 'language', 'voice', 'split']]
        model_key = (engine, model, speaker if engine == 'piper' else None)
        if model_key != key:
            if synthesizer is not None: del synthesizer
            if device != 'cpu': torch.cuda.empty_cache()
            synthesizer = Synthesizer(work, job, device, config.get('cpu_threads', 2))
            key = model_key
        synthesizer.job = job
        target = config['wake_word_ru' if language == 'ru' else 'wake_word_en'].strip()
        positives = [target + mark for mark in ['.', '!', '?']]
        if config.get('positive_bare_word'): positives.append(target)
        if not config['standalone_only']: positives += [f'{target}, привет.' if language == 'ru' else f'{target}, hello.']
        negatives = negative_texts(language, target, config['standalone_only'],
                                   config.get('negative_exclusions', {}).get(language, []))
        if config.get('negative_source_limit'):
            # Bounded development experiments retain every negative category.
            categories = [[t for t in negatives if negative_kind(t, target) == kind]
                          for kind in ['hard_word', 'context', 'ordinary']]
            negatives = [t for i in range(max(map(len, categories))) for category in categories for t in category[i:i+1]][:config['negative_source_limit']]
        if config.get('smoke'):
            negatives = [next(t for t in negatives if negative_kind(t, target) == kind)
                         for kind in ['hard_word', 'context', 'ordinary']] if config['standalone_only'] else negatives[:3]
        buckets = {kind: [] for kind in ['positive', 'hard_word', 'context', 'ordinary']}
        for label, texts in [(1, positives), (0, negatives)]:
            for text in texts:
                for variant in range(config.get('positive_tts_variants', 6) if label else (
                        config.get('negative_tts_variants', 1) if split == 'train' else 1)):
                    rate = ['normal', 'slow', 'fast'][variant % 3]
                    seed = stable_seed(f'{engine}:{model}:{speaker}:{text}:{variant}:{config["seed"]}')
                    source_id = hashlib.sha256(f'v2|{engine}|{model}|{speaker}|{text}|{rate}|{seed}'.encode()).hexdigest()[:24]
                    file = destination / 'sources' / (source_id + '.wav')
                    if not file.exists():
                        generated = synthesizer.say(text, rate, seed)
                        try: audio = prepare_source(generated)
                        except ValueError as error:
                            audit.append(dict(source_id=source_id, text=text, rate=rate, reason=str(error), speaker=job['speaker_group']))
                            continue
                        write_wav(file, audio)
                    audio = read_wav(file)
                    kind = 'positive' if label else negative_kind(text, target)
                    if len(audio) > N_SAMPLES - 480 and (label or kind == 'context'):
                        audit.append(dict(source_id=source_id, text=text, reason='overlong complete wake/context', speaker=job['speaker_group']))
                        continue
                    # Independent pitch shift, cached per source. Both classes get the same policy.
                    options = [audio]
                    if config.get('independent_pitch', True) and not config.get('smoke') and split == 'train':
                        import librosa
                        for steps in [-2., 2.]:
                            shifted_file = destination / 'sources' / f'{source_id}-pitch-{int(steps)}.wav'
                            if not shifted_file.exists():
                                shifted = librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=steps).astype(np.float32)
                                write_wav(shifted_file, shifted)
                            options.append(read_wav(shifted_file))
                    buckets[kind].append((source_id, text, options, kind))
        if not buckets['positive']: raise RuntimeError(f'No complete wake sources for {job}')
        multiplier = config.get('russian_multiplier', 2) if language == 'ru' else 1
        # Versions do not inflate the Russian sample budget; split it between renderings.
        if engine == 'silero' and language == 'ru': multiplier /= len(config.get('silero_ru_versions', ['v5_5_ru', 'v4_ru']))
        counts = {1: max(1, round(config['positive_per_voice'] * multiplier)),
                  0: max(1, round(config['negative_per_voice'] * multiplier))}
        if split != 'train' and not config.get('smoke'):
            counts = {1: max(1, round(config.get('eval_positive_per_voice', 200) * multiplier)),
                      0: max(1, round(config.get('eval_negative_per_voice', 800) * multiplier))}
        for label in [1, 0]:
            rng = np.random.default_rng(config['seed'] + stable_seed(f'{engine}:{model}:{speaker}:{label}'))
            written, attempts = 0, 0
            while written < counts[label]:
                attempts += 1
                if attempts > counts[label] * 40: raise RuntimeError(f'Too many rejected augmentations for {job}')
                if label: kind = 'positive'
                else:
                    available = [k for k in ['hard_word', 'context', 'ordinary'] if buckets[k]]
                    proportions = config.get('negative_kind_weights', {}) if split == 'train' else {}
                    weights = np.array([proportions.get(k, .5 if k == 'hard_word' else .25) for k in available])
                    kind = rng.choice(available, p=weights / weights.sum())
                source_id, text, variants, _ = buckets[kind][int(rng.integers(len(buckets[kind])))]
                audio = variants[int(rng.integers(len(variants)))]
                window = augment(audio, rng, label, kind == 'context', train=split == 'train', noise_pool=noise_pool)
                if window is None: continue
                identifier = f'{engine}-{model.split("/")[-1]}-{speaker}-{label}-{written:05d}'
                save(identifier, window, dict(label=label, split=split, language=language, speaker=f'{model}/{speaker}',
                     speaker_group=job['speaker_group'], source_id=source_id, text=text, generator=engine, kind=kind))
                written += 1
        print(f'[TTS {rank}] {engine}/{model}/{speaker} -> {split}: {sum(counts.values())} windows; provider={synthesizer.provider}', flush=True)
    for row in real[rank::world]:
        rng = np.random.default_rng(config['seed'] + stable_seed(row['id']))
        audio = read_wav(row['file'])
        try: audio = prepare_source(audio)
        except ValueError:
            # A quiet/empty negative word is still non-wake. Never relabel it positive.
            if row.get('label', 0) or not len(audio):
                audit.append(dict(source_id=row['id'], reason='empty real recording')); continue
        label = row.get('label', 0)
        human = row.get('generator') == 'human_reviewed'
        window = augment(audio, rng, label, preserve_context=human, train=row['split'] == 'train', noise_pool=noise_pool)
        if window is None and human:
            # A valid complete annotation may no longer fit after slowing it down.
            # Keep its original whole utterance instead of cropping or aborting.
            window = right_aligned_window(audio)
        if window is None: raise ValueError('Annotated human utterance cannot be cropped to preserve its label')
        save(row['id'], window, dict(label=label, split=row['split'], language=row.get('language', 'en'), speaker=row['speaker'],
             speaker_group=row['speaker_group'], source_id=row['id'], text=row['word'],
             generator=row.get('generator', 'real_speech'), kind='human_utterance' if human else 'real_word'))
    for index in range(rank, config['noise_windows'], world):
        rng = np.random.default_rng(config['seed'] + 100000 + index)
        identifier = f'noise-{index:05d}'
        save(identifier, background(rng, N_SAMPLES) * 10 ** (rng.uniform(-45, -12) / 20),
             dict(label=0, split=SPLITS[index % 4], language='none', speaker='noise', speaker_group='noise',
                  source_id=identifier, text='', generator='procedural_noise', kind='noise'))
    shard = destination / f'manifest-rank-{rank}.jsonl'
    temp = shard.with_suffix('.tmp')
    temp.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
    temp.replace(shard)
    atomic_json(work / f'source_audit-{rank}.json', audit)
    print(f'[TTS {rank}] finished {len(rows)} windows in {time.monotonic()-started:.1f}s', flush=True)


def merge_manifest(config, world):
    from wake_data import merge_manifest as merge
    rows, summary = merge(config, world)
    groups = {}
    for row in rows:
        if row.get('speaker_group', row['speaker']) != 'noise':
            identity = row.get('speaker_group', row['speaker'])
            assert groups.setdefault(identity, row['split']) == row['split'], 'Voice identity leakage across TTS versions'
    for split in SPLITS:
        selected = [r for r in rows if r['split'] == split and r['speaker'] != 'noise']
        summary[split]['voices'] = len({r.get('speaker_group', r['speaker']) for r in selected})
        summary[split]['tts_voice_groups'] = len({r['speaker_group'] for r in selected if r['generator'] in ['silero', 'piper', 'kokoro', 'mms']})
        summary[split]['tts_renderings'] = len({r['speaker'] for r in selected if r['generator'] in ['silero', 'piper', 'kokoro', 'mms']})
    atomic_json(Path(config['work_dir']) / 'dataset_summary.json', summary)
    details = {split: {language: {label: sum(r['split'] == split and r['language'] == language and r['label'] == label for r in rows)
                                 for label in [0, 1]} for language in ['ru', 'en', 'none']} for split in SPLITS}
    atomic_json(Path(config['work_dir']) / 'dataset_groups.json', details)
    return rows, summary


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--rank', type=int, default=0)
    parser.add_argument('--world', type=int, default=1)
    args = parser.parse_args()
    generate(json.loads(Path(args.config).read_text(encoding='utf-8')), args.rank, args.world)
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_metrics.py').write_text(r'''"""Threshold decisions use validation/calibration only, never test scores."""
import numpy as np


def confusion(labels, scores, threshold=.5):
    labels, scores = np.asarray(labels), np.asarray(scores)
    predictions, positive = scores >= threshold, labels == 1
    tp, fp = int(np.sum(predictions & positive)), int(np.sum(predictions & ~positive))
    fn, tn = int(np.sum(~predictions & positive)), int(np.sum(~predictions & ~positive))
    recall, precision = tp / max(1, tp + fn), tp / max(1, tp + fp)
    return dict(tp=tp, fp=fp, tn=tn, fn=fn, positive_windows=tp+fn, negative_windows=tn+fp,
                recall=recall, precision=precision, false_positive_rate_per_window=fp/max(1, fp+tn),
                false_negative_rate=fn/max(1, tp+fn), f1=2*precision*recall/max(1e-12, precision+recall))


def threshold_boundaries(labels, scores, target_fpr, languages=None, groups=None):
    """Expose each negative-tail boundary, including groups with no positives."""
    labels, scores = np.asarray(labels), np.asarray(scores, dtype=np.float64)
    if not 0 <= target_fpr < 1 or not np.isfinite(scores).all(): raise ValueError('Invalid calibration input')
    if labels.shape != scores.shape: raise ValueError('Calibration labels and scores must match')
    records = []
    def boundary(scope, group, selection):
        negatives = np.sort(scores[(labels == 0) & selection])[::-1]
        if not len(negatives): raise ValueError('Calibration requires negative labels')
        allowed = int(np.floor(target_fpr * len(negatives)))
        records.append(dict(scope=scope, group=group, negative_windows=len(negatives), allowed_fp=allowed,
                            threshold=float(np.nextafter(negatives[min(allowed, len(negatives)-1)], np.inf))))
    if not np.any(labels == 1): raise ValueError('Calibration requires both labels')
    boundary('overall', 'all', np.ones(len(labels), dtype=bool))
    # Prevent English/noise volume from hiding Russian false positives.
    if languages is not None:
        languages = np.asarray(languages)
        if languages.shape != labels.shape: raise ValueError('Calibration languages must match labels')
        for language in ['ru', 'en']:
            selection = languages == language
            if np.any((labels == 0) & selection): boundary('language', language, selection)
    if groups is not None:
        groups = np.asarray(groups)
        if groups.shape != labels.shape: raise ValueError('Calibration groups must match labels')
        for group in sorted(set(groups.tolist())):
            selection = groups == group
            if np.any((labels == 0) & selection): boundary('source', group, selection)
    return records


def calibrate(labels, scores, target_fpr, languages=None, groups=None):
    threshold = max(record['threshold'] for record in threshold_boundaries(labels, scores, target_fpr, languages, groups))
    return threshold, confusion(labels, scores, threshold)


def threshold_audit(labels, scores, target_fpr, languages, groups, constrain_sources=True):
    records = threshold_boundaries(labels, scores, target_fpr, languages, groups)
    for record in records: record['applied'] = record['scope'] != 'source' or constrain_sources
    threshold = max(record['threshold'] for record in records if record['applied'])
    for record in records: record['limits_threshold'] = record['applied'] and record['threshold'] == threshold
    return dict(threshold=threshold, source_constraints=constrain_sources, boundaries=records)


def source_group(row):
    """Keep Russian Piper and individual Silero versions visible in decisions."""
    generator = row['generator']
    rendering = row['speaker'].split('/')[0] if generator == 'silero' else generator
    return f'{row["language"]}|{generator}|{rendering}'


def operating_point(prediction, rows, target_fpr, by_language=True, by_source=False, constrain_sources=True, audit=False):
    languages = np.array([r['language'] for r in rows])
    labels, scores = np.asarray(prediction['labels']), np.asarray(prediction['scores'])
    sources = np.array([source_group(row) for row in rows]) if by_source else None
    threshold, metrics = calibrate(labels, scores, target_fpr, languages if by_language else None,
                                   sources if constrain_sources else None)
    recalls = []
    groups = {}
    for language in ['ru', 'en']:
        selected = languages == language
        if np.any((labels == 1) & selected):
            groups[language] = confusion(labels[selected], scores[selected], threshold)
            recalls.append(groups[language]['recall'])
    result = dict(threshold=threshold, metrics=metrics, groups=groups,
                  macro_recall=float(np.mean(recalls)), worst_language_recall=min(recalls))
    if by_source:
        source_metrics = {group: confusion(labels[sources == group], scores[sources == group], threshold)
                          for group in sorted(set(sources.tolist()))}
        positive_groups = [item['recall'] for item in source_metrics.values() if item['positive_windows']]
        result.update(source_groups=source_metrics, worst_source_recall=min(positive_groups))
    if audit:
        result['threshold_audit'] = threshold_audit(labels, scores, target_fpr, languages if by_language else None,
                                                   sources, constrain_sources)
    return result


def point_for_config(prediction, rows, config):
    return operating_point(prediction, rows, config['target_fpr'], by_source=config.get('source_groups', False),
                           constrain_sources=config.get('source_group_fpr_constraint', True),
                           audit=config.get('threshold_diagnostics', False))


def selection_score(point, loss, config):
    score = point['macro_recall'] + .25 * point['worst_language_recall'] - .05 * loss
    if config.get('source_groups'): score += .25 * point['worst_source_recall']
    return score
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_train.py').write_text(r'''"""torchrun entry point: train, validation and mining share all available ranks."""
import argparse
import copy
import contextlib
from datetime import timedelta
import hashlib
import json
import os
from pathlib import Path
import random
import time
import numpy as np
import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler, Sampler, Subset
from wake_model import make_model, spec_augment
from wake_data import atomic_json
from wake_metrics import point_for_config, selection_score


class FeatureDataset(Dataset):
    def __init__(self, root, split, preload=False):
        self.root = Path(root)
        self.rows = [json.loads(line) for line in (self.root / 'manifest.jsonl').read_text(encoding='utf-8').splitlines()]
        self.rows = [row for row in self.rows if row['split'] == split]
        if not self.rows: raise ValueError(f'Empty {split} split')
        self.cached = np.stack([np.load(self.root / row['feature']) for row in self.rows]) if preload else None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        feature = (self.cached[index] if self.cached is not None else np.load(self.root / row['feature'])).astype(np.float32)
        return torch.from_numpy(feature), torch.tensor(row['label'], dtype=torch.float32)


class BalancedSampler(Sampler):
    """One deterministic weighted stream, partitioned equally across DDP ranks."""
    def __init__(self, rows, seed, rank=0, world=1):
        self.rows, self.seed, self.rank, self.world, self.epoch = rows, seed, rank, world, 0
        self.keys = [(r['language'], r['label'], r['generator']) for r in rows]
        counts = {}
        for key in self.keys: counts[key] = counts.get(key, 0) + 1
        groups = {}
        for language, label, engine in counts: groups[(language, label)] = groups.get((language, label), 0) + 1
        weights = []
        for key in self.keys:
            language, label, _ = key
            language_mass = {'ru': .7, 'en': .3}.get(language, 0) if label else {'ru': .6, 'en': .3, 'none': .1}.get(language, .1)
            weights.append(language_mass / (counts[key] * groups[(language, label)]))
        self.base = torch.tensor(weights, dtype=torch.double)
        for label in [0, 1]:
            selected = torch.tensor([r['label'] == label for r in rows])
            self.base[selected] *= .5 / self.base[selected].sum()
        self.weights = self.base.clone()
        self.size = (len(rows) + world - 1) // world

    def set_epoch(self, epoch): self.epoch = epoch
    def __len__(self): return self.size
    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        indices = torch.multinomial(self.weights, self.size * self.world, replacement=True, generator=generator).tolist()
        return iter(indices[self.rank::self.world])

    def mine(self, scores):
        scores = np.asarray(scores)
        difficulty = torch.tensor([1 + min(2., 2 * (1 - s if r['label'] else s)) for r, s in zip(self.rows, scores, strict=True)])
        self.weights = self.base * difficulty
        # Mining changes examples within groups, while preserving language/class/engine masses.
        for key in set(self.keys):
            selected = torch.tensor([k == key for k in self.keys])
            self.weights[selected] *= self.base[selected].sum() / self.weights[selected].sum()


def confusion(labels, scores, threshold=0.5):
    labels, scores = np.asarray(labels), np.asarray(scores)
    predictions = scores >= threshold
    positive, negative = labels == 1, labels == 0
    tp, fp = int(np.sum(predictions & positive)), int(np.sum(predictions & negative))
    fn, tn = int(np.sum(~predictions & positive)), int(np.sum(~predictions & negative))
    recall = tp / max(1, tp + fn)
    precision = tp / max(1, tp + fp)
    return {'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn, 'positive_windows': tp + fn, 'negative_windows': tn + fp,
            'recall': recall, 'precision': precision, 'false_positive_rate_per_window': fp / max(1, fp + tn),
            'false_negative_rate': fn / max(1, tp + fn), 'f1': 2 * precision * recall / max(1e-12, precision + recall)}


@torch.inference_mode()
def evaluate(model, loader, device, amp=False):
    model.eval()
    labels, scores, logits = [], [], []
    loss_sum = 0
    for batch in loader:
        feature, label = batch[:2]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == 'cuda'):
            output = model(feature.to(device, non_blocking=True))
        output = output.float()
        loss_sum += float(nn.functional.binary_cross_entropy_with_logits(output, label.to(device), reduction='sum'))
        logits.extend(output.cpu().tolist())
        scores.extend(output.sigmoid().cpu().tolist())
        labels.extend(label.tolist())
    return {'labels': labels, 'scores': scores, 'logits': logits, 'loss': loss_sum / max(1, len(labels))}


def merge_evaluations(shards, size):
    """Restore manifest order and reject padded, missing or duplicated evaluation rows."""
    records, loss_sum = {}, 0.
    for shard in shards:
        prediction, indices = shard['prediction'], shard['indices']
        loss_sum += prediction['loss'] * len(indices)
        for index, label, score, logit in zip(indices, prediction['labels'], prediction['scores'], prediction['logits'], strict=True):
            if index in records or not 0 <= index < size:
                raise ValueError('Evaluation shards contain duplicate/out-of-range rows')
            records[index] = (label, score, logit)
    if len(records) != size:
        raise ValueError('Evaluation shards do not cover the dataset')
    return dict(labels=[records[i][0] for i in range(size)], scores=[records[i][1] for i in range(size)],
                logits=[records[i][2] for i in range(size)], loss=loss_sum / max(1, size))


def evaluate_shared(model, dataset, batch_size, device, amp=False):
    """Every rank evaluates an unpadded shard using its raw model, then gathers CPU results."""
    distributed = dist.is_available() and dist.is_initialized()
    rank, world = (dist.get_rank(), dist.get_world_size()) if distributed else (0, 1)
    indices = list(range(rank, len(dataset), world))
    prediction = evaluate(model, DataLoader(Subset(dataset, indices), batch_size=batch_size, num_workers=0,
                                           pin_memory=device.type == 'cuda'), device, amp=amp)
    shard = dict(indices=indices, prediction=prediction)
    shards = [shard]
    if distributed:
        shards = [None] * world
        dist.all_gather_object(shards, shard)
    return merge_evaluations(shards, len(dataset))


def save_torch(file, value):
    temporary = file.with_suffix('.tmp')
    torch.save(value, temporary)
    temporary.replace(file)


def train(config, member=0):
    config = dict(config)
    config.update(config.get('member_overrides', {}).get(str(member), {}))
    rank, local_rank, world = int(os.environ.get('RANK', 0)), int(os.environ.get('LOCAL_RANK', 0)), int(os.environ.get('WORLD_SIZE', 1))
    cuda = torch.cuda.is_available() and not config.get('force_cpu')
    device = torch.device(f'cuda:{local_rank}' if cuda else 'cpu')
    distributed = world > 1
    if cuda: torch.cuda.set_device(local_rank)
    if distributed:
        dist.init_process_group('nccl' if cuda else 'gloo', timeout=timedelta(seconds=config.get('ddp_timeout_seconds', 1800)))
    try:
        torch.set_num_threads(config.get('cpu_threads', 2))
        seed = config['seed'] + member * config.get('member_seed_stride', 1009)
        torch.manual_seed(seed + rank)
        random.seed(seed + rank)
        np.random.seed(seed + rank)
        if cuda: torch.cuda.manual_seed_all(seed + rank)
        torch.backends.cudnn.benchmark = False
        work = Path(config['work_dir'])
        multiple = config.get('ensemble_members', 1) > 1
        tag = f'-member-{member}' if multiple else ''
        checkpoints = work / 'checkpoints' / f'member-{member}' if multiple else work / 'checkpoints'
        checkpoints.mkdir(parents=True, exist_ok=True)
        dataset_root = Path(config.get('dataset_dir') or work / 'dataset')
        dataset = FeatureDataset(dataset_root, 'train', config.get('preload_features', False))
        if config.get('distill_weight', 0):
            from wake_distill import DistillationDataset
            dataset = DistillationDataset(dataset, config['teacher_targets'])
        validation = FeatureDataset(dataset_root, 'validation', config.get('preload_features', False))
        # Padding at the sampler is intentional; every rank executes the same number of steps.
        sampler = BalancedSampler(dataset.rows, seed, rank, world) if config.get('balanced_sampling') else (
            DistributedSampler(dataset, num_replicas=world, rank=rank, shuffle=True, seed=seed) if distributed else None)
        generator = torch.Generator().manual_seed(seed + rank)
        loader = DataLoader(dataset, batch_size=config['batch_per_gpu'], sampler=sampler, shuffle=sampler is None,
                            num_workers=config['loader_workers'], pin_memory=cuda, drop_last=False, generator=generator)
        model = make_model(config).to(device)
        if distributed: model = DDP(model, device_ids=[local_rank] if cuda else None, broadcast_buffers=False)
        raw_model = model.module if distributed else model
        ema = copy.deepcopy(raw_model).eval().requires_grad_(False) if config.get('ema', False) else None
        ema_steps = 0
        if config.get('architecture') == 'whisper_encoder_v4':
            backbone = [p for name, p in model.named_parameters() if 'encoder.' in name and p.requires_grad]
            head = [p for name, p in model.named_parameters() if 'encoder.' not in name and p.requires_grad]
            optimizer = torch.optim.AdamW([dict(params=backbone, lr=config.get('encoder_learning_rate', 1e-5)),
                                           dict(params=head, lr=config['learning_rate'])], weight_decay=1e-3)
        else:
            optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])
        amp = cuda and config.get('mixed_precision', True)
        scaler = torch.amp.GradScaler('cuda', enabled=amp)
        positive = sum(row['label'] == 1 for row in dataset.rows)
        negative = len(dataset) - positive
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(1. if config.get('balanced_sampling') else negative / positive, device=device))
        manifest_hash = hashlib.sha256((dataset_root / 'manifest.jsonl').read_bytes()).hexdigest()
        recipe_keys = ['seed', 'epochs', 'batch_per_gpu', 'learning_rate', 'architecture', 'balanced_sampling', 'ema', 'mining_interval', 'target_fpr', 'encoder_learning_rate', 'encoder_train_layers']
        recipe_keys += [key for key in ['encoder_dir', 'distill_weight', 'distill_temperature', 'member_seed_stride', 'source_groups', 'source_group_fpr_constraint'] if key in config]
        if config.get('distill_weight', 0):
            config['teacher_targets_sha256'] = hashlib.sha256(Path(config['teacher_targets']).read_bytes()).hexdigest()
            recipe_keys.append('teacher_targets_sha256')
        training_recipe_hash = hashlib.sha256(json.dumps({k: config.get(k) for k in recipe_keys}, sort_keys=True).encode()).hexdigest()
        start_epoch, best_loss, history, best_score, stale = 0, float('inf'), [], -float('inf'), 0
        resume = config.get('resume_checkpoint')
        if resume and multiple: resume = str(Path(resume) / f'member-{member}' / 'latest.pt')
        if resume:
            saved = torch.load(resume, map_location='cpu', weights_only=False)
            if saved['manifest_sha256'] != manifest_hash or saved['world_size'] != world:
                raise ValueError('Resume requires the same dataset and number of processes')
            if saved.get('training_recipe_sha256', training_recipe_hash) != training_recipe_hash:
                raise ValueError('Resume requires the same training recipe, including total epochs')
            raw_model.load_state_dict(saved['model'])
            optimizer.load_state_dict(saved['optimizer'])
            scheduler.load_state_dict(saved['scheduler'])
            scaler.load_state_dict(saved['scaler'])
            start_epoch, best_loss, history = saved['epoch'] + 1, saved['best_loss'], saved['history']
            if saved.get('architecture', 'legacy') != config.get('architecture', 'legacy') or saved.get('member', 0) != member:
                raise ValueError('Resume requires the same architecture and ensemble member')
            best_score, stale = saved.get('best_score', -float('inf')), saved.get('stale', 0)
            if ema is not None:
                ema.load_state_dict(saved['ema'])
                ema_steps = saved['ema_steps']
            if isinstance(sampler, BalancedSampler) and saved.get('sampler_weights') is not None:
                sampler.weights = saved['sampler_weights']
            state = torch.load(Path(resume).with_name(f'latest-rng-{rank}.pt'), weights_only=False, map_location='cpu')
            torch.set_rng_state(state['torch'])
            generator.set_state(state['loader'])
            if cuda: torch.cuda.set_rng_state(state['cuda'], device=device)
            # best weights are embedded so a copied latest.pt can restore them in the new output directory.
            if rank == 0: save_torch(checkpoints / 'best.pt', saved['best'])
        if rank == 0:
            print(f'Training: {world} processes, {device.type}, effective batch {config["batch_per_gpu"] * world}, '
                  f'{sum(p.numel() for p in raw_model.parameters()):,} parameters, {len(dataset)} train windows', flush=True)
        best = saved['best'] if resume else None
        end_epoch = min(config['epochs'], config.get('stop_after_epoch') or config['epochs'])
        for epoch in range(start_epoch, end_epoch):
            started = time.monotonic()
            if sampler is not None: sampler.set_epoch(epoch)
            model.train()
            totals = torch.zeros(2, dtype=torch.float64, device=device)
            for batch in loader:
                feature, label = batch[:2]
                feature = spec_augment(feature.to(device, non_blocking=True))
                label = label.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                    output = model(feature)
                    loss = criterion(output, label)
                    if config.get('distill_weight', 0):
                        from wake_distill import distillation_loss
                        teacher_logit = batch[2].to(device, non_blocking=True)
                        loss = loss + config['distill_weight'] * distillation_loss(
                            output, teacher_logit, label, config.get('distill_temperature', 2.))
                if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5)
                scaler.step(optimizer)
                scaler.update()
                if ema is not None:
                    ema_steps += 1
                    decay = min(.995, (1 + ema_steps) / (10 + ema_steps))
                    with torch.no_grad():
                        for averaged, parameter in zip(ema.parameters(), raw_model.parameters(), strict=True):
                            averaged.lerp_(parameter.detach(), 1 - decay)
                totals[0] += loss.detach().double() * len(label)
                totals[1] += len(label)
            scheduler.step()
            if distributed: dist.all_reduce(totals)
            if rank == 0: print(f'Epoch {epoch + 1}: validation on {world} rank(s)', flush=True)
            validation_started = time.monotonic()
            result = evaluate_shared(raw_model, validation, config['batch_per_gpu'] * 2, device, amp=amp)
            averaged_result = evaluate_shared(ema, validation, config['batch_per_gpu'] * 2, device, amp=amp) if ema is not None else None
            if rank == 0:
                chosen, source = raw_model, 'raw'
                point = point_for_config(result, validation.rows, config) if config.get('balanced_sampling') else None
                selection = selection_score(point, result['loss'], config) if point else -result['loss']
                if ema is not None:
                    averaged_point = point_for_config(averaged_result, validation.rows, config)
                    averaged_selection = selection_score(averaged_point, averaged_result['loss'], config)
                    if averaged_selection > selection:
                        result, point, selection, chosen, source = averaged_result, averaged_point, averaged_selection, ema, 'ema'
                row = {'epoch': epoch + 1, 'train_loss': float(totals[0] / totals[1]),
                       'validation_loss': result['loss'], 'seconds': time.monotonic() - started,
                       'validation_seconds': time.monotonic() - validation_started, 'evaluation_world_size': world,
                       **confusion(result['labels'], result['scores'])}
                if point: row.update(validation_operating_point=point, selection_score=selection, weights_source=source)
                history.append(row)
                improved = selection > best_score + 1e-5
                stale = 0 if improved else stale + 1
                if improved:
                    best_loss = result['loss']
                    best_score = selection
                    best = {'model': {key: value.detach().cpu().clone() for key, value in chosen.state_dict().items()},
                            'epoch': epoch + 1, 'validation_loss': best_loss, 'selection_score': best_score,
                            'architecture': config.get('architecture', 'legacy'), 'member': member, 'weights_source': source}
                    save_torch(checkpoints / 'best.pt', best)
            # Mine training examples only. All ranks receive the same new sampling stream.
            if isinstance(sampler, BalancedSampler) and (epoch + 1) % config.get('mining_interval', 10) == 0:
                if rank == 0: print(f'Epoch {epoch + 1}: train-only mining on {world} rank(s)', flush=True)
                mined = evaluate_shared(raw_model, dataset, config['batch_per_gpu'] * 2, device, amp=amp)
                sampler.mine(mined['scores'])
            if rank == 0 and config.get('persist_latest', True):
                save_torch(checkpoints / 'latest.pt', {'model': raw_model.state_dict(), 'optimizer': optimizer.state_dict(),
                           'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoch,
                           'best_loss': best_loss, 'best_score': best_score, 'stale': stale, 'best': best, 'history': history,
                           'ema': ema.state_dict() if ema is not None else None, 'ema_steps': ema_steps,
                           'sampler_weights': sampler.weights if isinstance(sampler, BalancedSampler) else None,
                           'architecture': config.get('architecture', 'legacy'), 'member': member,
                           'training_recipe_sha256': training_recipe_hash,
                           'manifest_sha256': manifest_hash, 'world_size': world})
            if rank == 0:
                atomic_json(work / f'training_history{tag}.json', history)
                print(f'Epoch {epoch + 1}/{config["epochs"]}: train {row["train_loss"]:.4f}, '
                      f'validation {row["validation_loss"]:.4f}, recall {row["recall"]:.3f}, '
                      f'FP/window {row["false_positive_rate_per_window"]:.4f}', flush=True)
                if point: print(f'  Validation at target FPR: macro recall {point["macro_recall"]:.3f}, worst language {point["worst_language_recall"]:.3f}, weights {source}', flush=True)
            if config.get('persist_latest', True):
                save_torch(checkpoints / f'latest-rng-{rank}.pt', {'torch': torch.get_rng_state(), 'loader': generator.get_state(),
                           'cuda': torch.cuda.get_rng_state(device) if cuda else None})
            if distributed: dist.barrier()
            stop = torch.tensor(int(rank == 0 and stale >= config.get('patience', config['epochs'] + 1)
                                    and epoch + 1 >= config.get('min_epochs', 15)), device=device)
            if distributed: dist.broadcast(stop, src=0)
            if stop.item():
                if rank == 0: print('Early stop: no validation operating-point improvement', flush=True)
                break
        if not (checkpoints / 'best.pt').exists(): raise RuntimeError('No trained checkpoint to export')
        if rank == 0:
            run = {'world_size': world, 'device': str(device), 'amp': amp,
                        'manifest_sha256': manifest_hash, 'pytorch': torch.__version__,
                        'epochs_completed': len(history), 'smoke_only': config.get('smoke', False),
                        'architecture': config.get('architecture', 'legacy'), 'member': member,
                        'ensemble_members': config.get('ensemble_members', 1),
                        'evaluation_world_size': world, 'ddp_timeout_seconds': config.get('ddp_timeout_seconds', 1800),
                        'distill_weight': config.get('distill_weight', 0), 'encoder_dir': config.get('encoder_dir')}
            atomic_json(work / f'training_run{tag}.json', run)
            atomic_json(work / 'training_run.json', run)
    finally:
        if distributed and dist.is_initialized(): dist.destroy_process_group()


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--member', type=int, default=0)
    args = parser.parse_args()
    train(json.loads(Path(args.config).read_text(encoding='utf-8')), args.member)
''', encoding='utf-8', newline='\n')


In [ ]:
(CODE_DIR / 'wake_export.py').write_text(r'''"""Freeze calibration threshold, evaluate untouched test, check CPU ONNX parity."""
import argparse
import hashlib
import json
from pathlib import Path
import shutil
import time
import numpy as np
import torch
from torch.utils.data import DataLoader
from wake_audio import SAMPLE_RATE, N_SAMPLES, N_MELS, N_FFT, HOP, make_frontend, read_wav
from wake_model import make_model, ProbabilityModel, EnsembleLogits
from wake_train import FeatureDataset, evaluate, confusion
from wake_data import atomic_json
from wake_metrics import calibrate as calibrate, operating_point


def export(config):
    import onnx
    import onnxruntime as ort
    torch.set_num_threads(config.get('cpu_threads', 2))
    work = Path(config['work_dir'])
    members = config.get('ensemble_members', 1)
    models, bests = [], []
    for member in range(members):
        directory = work / 'checkpoints' / f'member-{member}' if members > 1 else work / 'checkpoints'
        best = torch.load(directory / 'best.pt', map_location='cpu', weights_only=False)
        model = make_model(config)
        model.load_state_dict(best['model'])
        models.append(model.eval())
        bests.append(best)
    selected_members, selection_report = list(range(members)), []
    if members > 1:
        validation = FeatureDataset(work / 'dataset', 'validation')
        candidates = [([i], model) for i, model in enumerate(models)]
        if config.get('allow_ensemble', True):
            candidates += [(list(range(members)), EnsembleLogits(models).eval())]
        highest = -float('inf')
        for indices, candidate in candidates:
            result = evaluate(candidate, DataLoader(validation, batch_size=128), torch.device('cpu'))
            point = operating_point(result, validation.rows, config['target_fpr'])
            score = point['macro_recall'] + .25 * point['worst_language_recall'] - .05 * result['loss']
            selection_report.append(dict(members=indices, score=score, validation=point))
            if score > highest:
                highest, selected_members = score, indices
        atomic_json(work / 'ensemble_selection.json', dict(selected_members=selected_members, candidates=selection_report,
                    decision_data='validation only; calibration/test not used'))
    chosen_models = [models[i] for i in selected_members]
    model = chosen_models[0] if len(chosen_models) == 1 else EnsembleLogits(chosen_models).eval()
    predictions = {}
    for split in ['calibration', 'test']:
        dataset = FeatureDataset(work / 'dataset', split)
        predictions[split] = (dataset, evaluate(model, DataLoader(dataset, batch_size=128, num_workers=0), torch.device('cpu')))
    test_dataset, test_predictions = predictions['test']
    smoke = config.get('smoke', False)
    output = work / ('wake-model-SMOKE-ONLY' if smoke else 'wake-model')
    output.mkdir(parents=True, exist_ok=True)
    probability_model = ProbabilityModel(model).eval()
    probe = torch.randn(3, *test_dataset[0][0].shape)
    onnx_file = output / 'wake_model.onnx'
    torch.onnx.export(probability_model, (probe,), str(onnx_file), input_names=['log_mel'], output_names=['probability'],
                      dynamic_axes={'log_mel': {0: 'batch'}, 'probability': {0: 'batch'}}, opset_version=17, dynamo=False)
    onnx.checker.check_model(str(onnx_file))
    options = ort.SessionOptions()
    options.intra_op_num_threads = 2; options.inter_op_num_threads = 1
    session = ort.InferenceSession(str(onnx_file), options, providers=['CPUExecutionProvider'])
    covered, validation_rows = set(), []
    for row in test_dataset.rows:
        group = (row['generator'], row['language'], row['label'])
        if group not in covered:
            covered.add(group); validation_rows.append(row)
    frontend = make_frontend(config, work / ('whisper-encoder' if config.get('architecture') == 'whisper_encoder_v4' else 'speech-embedding'))
    tensors = np.stack([frontend(read_wav(work / 'dataset' / row['wav'])) for row in validation_rows])
    cached_tensors = np.stack([np.load(work / 'dataset' / row['feature']) for row in validation_rows])
    np.testing.assert_allclose(tensors, cached_tensors, rtol=1e-5, atol=1e-6)
    with torch.inference_mode(): expected = probability_model(torch.from_numpy(tensors)).numpy()
    actual = session.run(['probability'], {'log_mel': tensors})[0]
    np.testing.assert_allclose(actual, expected, rtol=1e-4, atol=1e-5)
    # Calibrate the actual exported runtime, using the same FP32 frontend as inference.
    for dataset, prediction in predictions.values():
        scores = []
        for feature, _ in DataLoader(dataset, batch_size=128, num_workers=0):
            scores.extend(session.run(['probability'], {'log_mel': feature.numpy()})[0].tolist())
        np.testing.assert_allclose(scores, prediction['scores'], rtol=1e-4, atol=1e-5)
        prediction['scores'] = scores
    calibration = predictions['calibration'][1]
    calibration_languages = [row['language'] for row in predictions['calibration'][0].rows] if config.get('balanced_sampling') else None
    threshold, calibration_metrics = calibrate(calibration['labels'], calibration['scores'], config['target_fpr'], calibration_languages)
    test_metrics = confusion(test_predictions['labels'], test_predictions['scores'], threshold)
    if config.get('export_pytorch_weights', True):
        torch.save(dict(architecture=config.get('architecture', 'legacy'), selected_members=selected_members,
                       models=[m.state_dict() for m in chosen_models]), output / 'wake_model_state.pt')
    for file in ['wake_audio.py', 'wake_model.py']:
        shutil.copy2(Path(__file__).parent / file, output / file)
    metadata = {'schema_version': 1, 'wake_word_ru': config['wake_word_ru'], 'wake_word_en': config['wake_word_en'] if config['use_english'] else None,
                'task': 'isolated_wake_word' if config['standalone_only'] else 'keyword_presence_in_window',
                'standalone_only': config['standalone_only'], 'threshold': threshold, 'sample_rate': SAMPLE_RATE,
                'window_samples': N_SAMPLES, 'input': 'log_mel float32 [batch,1,40,201]', 'output': 'probability float32 [batch]',
                'frontend': {'n_fft': N_FFT, 'hop': HOP, 'n_mels': N_MELS, 'fmin': 50, 'fmax': 7600,
                             'window': 'periodic Hann', 'center_pad': 'zero', 'normalization': 'per window mean/std'},
                'smoke_only': smoke, 'quality_validated_on_real_users': False,
                'threshold_allows_any_detection': threshold <= 1,
                'best_epoch': bests[selected_members[0]]['epoch'], 'best_epochs': [b['epoch'] for b in bests],
                'architecture': config.get('architecture', 'legacy'), 'selected_members': selected_members,
                'calibration_fpr_constraints': 'overall + Russian + English' if calibration_languages else 'overall',
                'model_sha256': hashlib.sha256(onnx_file.read_bytes()).hexdigest()}
    if config.get('architecture') == 'speech_embedding_v3':
        metadata['input'] = 'log_mel (legacy name): speech embeddings float32 [batch,1,96,16]'
        metadata['frontend'] = dict(type='google_speech_embedding_oww_v0.5.1', pcm_scale=32768,
                                    mel_transform='mel / 10 + 2', patch_frames=76, patch_step=8, full_window=True)
        metadata['frontend_sha256'] = {}
        for name in ['melspectrogram.onnx', 'embedding_model.onnx', 'provenance.json']:
            shutil.copy2(work / 'speech-embedding' / name, output / name)
            metadata['frontend_sha256'][name] = hashlib.sha256((output / name).read_bytes()).hexdigest()
    if config.get('architecture') == 'whisper_encoder_v4':
        metadata['input'] = 'log_mel float32 [batch,1,80,200]'
        metadata['frontend'] = dict(type='whisper_log_mel_2s', n_fft=400, hop=160, n_mels=80,
                                    center_pad='reflect', normalization='log10 dynamic range /4 +1', encoder_positions=100)
        metadata['frontend_sha256'] = {}
        for name in ['whisper_mel_filters.npy', 'provenance.json']:
            shutil.copy2(work / 'whisper-encoder' / name, output / name)
            metadata['frontend_sha256'][name] = hashlib.sha256((output / name).read_bytes()).hexdigest()
    atomic_json(output / 'wake_config.json', metadata)
    by_group = {}
    for field in ['language', 'generator', 'speaker', *(['kind'] if 'kind' in test_dataset.rows[0] else [])]:
        by_group[field] = {}
        for value in sorted({row[field] for row in test_dataset.rows}):
            indices = [i for i, row in enumerate(test_dataset.rows) if row[field] == value]
            by_group[field][value] = confusion([test_predictions['labels'][i] for i in indices],
                                              [test_predictions['scores'][i] for i in indices], threshold)
    # Windows overlap/correlate; these figures are NOT false activations per real-world hour.
    report = {'source': 'synthetic neural TTS + procedural noise', 'calibration_threshold': threshold,
              'target_fpr_per_window': config['target_fpr'], 'calibration': calibration_metrics, 'test': test_metrics,
              'test_groups': by_group, 'onnx_max_absolute_difference': float(np.max(np.abs(actual - expected))),
              'real_audio_evaluation': 'real negative words only' if any(r['generator'] == 'real_speech' for r in test_dataset.rows) else 'not performed',
              'smoke_only': smoke,
              'interpretation': 'Speaker-held-out windows; augmentation siblings are correlated. No real-world FAR/hour claim.',
              'architecture': config.get('architecture', 'legacy'), 'selected_members': selected_members,
              'ensemble_selection': selection_report, 'real_wake_positive_evaluation': 'not performed'}
    human_indices = [i for i, row in enumerate(test_dataset.rows) if row['generator'] == 'human_reviewed']
    if human_indices:
        report['reviewed_human_test'] = confusion([test_predictions['labels'][i] for i in human_indices],
                                                 [test_predictions['scores'][i] for i in human_indices], threshold)
        if report['reviewed_human_test']['positive_windows']:
            report['real_wake_positive_evaluation'] = 'reviewed human test recordings; see reviewed_human_test counts'
    if config.get('balanced_sampling'):
        report['source'] = 'Four local TTS families + procedural noise + optional real negative speech'
        report['threshold_profiles'] = []
        for target in [.002, .005, .01, .02]:
            boundary, metrics = calibrate(calibration['labels'], calibration['scores'], target, calibration_languages)
            report['threshold_profiles'].append(dict(target_fpr=target, threshold=boundary, calibration=metrics,
                test=confusion(test_predictions['labels'], test_predictions['scores'], boundary)))
        report['profiles_interpretation'] = 'All FPR targets fixed before evaluation. Primary target remains run_config.target_fpr; do not select using test.'
    # Publish auditable test scores, not just aggregate accuracy.
    records = [{'id': row['id'], 'speaker': row['speaker'], 'language': row['language'], 'label': row['label'],
                'generator': row['generator'], 'kind': row.get('kind'), 'text': row.get('text'),
                'probability': score, 'predicted_wake': score >= threshold} for row, score in zip(test_dataset.rows, test_predictions['scores'], strict=True)]
    atomic_json(work / 'test_predictions.json', records)
    atomic_json(output / 'test_predictions.json', records)
    session.run(['probability'], {'log_mel': tensors[:1]})
    started = time.perf_counter()
    for _ in range(100): session.run(['probability'], {'log_mel': tensors[:1]})
    report['cpu_onnx_ms_per_window_excluding_frontend'] = (time.perf_counter() - started) * 10
    if config.get('architecture') in ['speech_embedding_v3', 'whisper_encoder_v4']:
        started = time.perf_counter()
        audio = read_wav(work / 'dataset' / validation_rows[0]['wav'])
        for _ in range(20): session.run(['probability'], {'log_mel': frontend(audio)[None]})
        report['cpu_ms_per_window_including_pretrained_frontend'] = (time.perf_counter()-started) * 50
    atomic_json(output / 'training_report.json', report)
    if members > 1:
        histories = []
        for i in range(members):
            histories += [dict(member=i, **row) for row in json.loads((work / f'training_history-member-{i}.json').read_text(encoding='utf-8'))]
        atomic_json(work / 'training_history.json', histories)
    for file in ['training_history.json', 'training_run.json', 'dataset_summary.json', 'tts_provenance.json', 'run_config.json',
                 'ensemble_selection.json', 'real_provenance.json', 'dataset_groups.json']:
        if not (work / file).exists(): continue
        shutil.copy2(work / file, output / file)
    for file in work.glob('source_audit-*.json'): shutil.copy2(file, output / file.name)
    (output / 'README.txt').write_text(
        'Binary acoustic wake-word detector. Mono PCM16/float32, 16000 Hz, 2-second window.\n'
        'pip install numpy onnxruntime\npython wake_audio.py . example.wav\n'
        'This training uses synthetic speech. Measure recall and false alarms on real microphones before deployment.\n'
        'The bot integration is separate; this export does not modify the Discord bot.\n'
        'TTS sources and licence references are in tts_provenance.json.\n', encoding='utf-8')
    archive = shutil.make_archive(str(output), 'zip', root_dir=output)
    print(json.dumps({'archive': archive, 'test': test_metrics, 'threshold': threshold,
                      'onnx_parity': report['onnx_max_absolute_difference'], 'smoke_only': smoke}, indent=2), flush=True)
    return output, report


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    args = parser.parse_args()
    export(json.loads(Path(args.config).read_text(encoding='utf-8')))
''', encoding='utf-8', newline='\n')


In [ ]:
SOURCE_SHA256 = {'wake_audio.py': 'dd24148cfbc1ff8480191c5184632cf9d0caffe5dfc9e7d37a4bfadb665bf75e', 'wake_model.py': '0d6a19f8c2635d4db3b1b3f30f94ab770b2a16011c982f11f826a4876fdc310a', 'wake_data.py': '15a56133d35dbd14f0b07757bb08f92204f25d7baa42efa520005adc9a1ded5d', 'wake_tts.py': '44827455441ebca700e3cb1abd989fc5dc46885386d33f76f8b0db7912e61a3a', 'wake_data_v2.py': '59be3d87e4f6cd2f8648166daa72676fcfa1d2113970e20ef1bfd6afabc3e915', 'wake_metrics.py': '65c17319ef1c7fa990c6586d79cdda8914a72ce8ea7c998c3d7914c819109873', 'wake_train.py': 'b1195f165a3e4990ccde4040361762ba1184b52af1d598bb597776f17f61eb3d', 'wake_export.py': '6e6760c7b41647d4c60d5b3a0e2a3a86a5b83185af587c00efba3f9af5b7bbf8'}
for name, expected in SOURCE_SHA256.items():
    assert hashlib.sha256((CODE_DIR / name).read_bytes()).hexdigest() == expected
(WORK_DIR / 'source_sha256.json').write_text(json.dumps(SOURCE_SHA256, indent=2))
sys.path.insert(0, str(CODE_DIR))
import wake_data_v2 as wake_data, wake_audio
print('Embedded modules ready; no Git clone required.')


## 4. Нейросетевая синтетика на двух GPU

**Silero v4/v5.5/v3_en + Piper + MMS + Kokoro**: 66 наборов TTS/голос,
61 группа голоса. Разные версии Silero с одним голосом остаются в одной части;
версии и изменение pitch не выдаются за новых людей. Русский train включает
2 голоса Silero, 3 Piper и MMS вместо прежних двух голосов одного синтезатора.
Женский голос Piper находится в train: validation выявил плохой перенос,
когда в train этого движка были только мужские голоса. Baya остаётся в validation.
Все голоса назначаются в train/validation/calibration/test до аугментаций.
Вспомогательный Speech Commands даёт реальные негативные слова и фоновые записи.
По умолчанию полный архив занимает около 2.4 GB; для малого запуска есть `mini`.
Дикторы реальных негативов тоже не пересекаются между частями.

Положительные примеры меняют интонацию и темп TTS. Отрицательные включают
«вот», «кот», «год», «борт», «порт», “boat”, “boot”, “but”, обычные фразы,
команды и контекстные упоминания. Шум, реверберация, уровень сигнала,
изменение темпа и независимый pitch shift применяются к обоим классам.
Среди речевых негативов выделены 50% коротких слов, 25% упоминаний и 25% фраз.
Есть чистые примеры, умеренный SNR, saturation/quantization и полоса 8 кГц.
Реальные негативные слова не заменяют реальные положительные обращения и музыку.

Два процесса делят голоса: Silero/MMS используют соответствующую GPU,
Piper/Kokoro — CPU ONNX с ограничением потоков. Генерация не заявляет ускорение
каждой модели на CUDA. При отсутствии выбранного TTS запуск завершается ошибкой.
Скачивание весов производится заранее один раз. Промежуточные WAV сохранены,
поэтому повторный запуск использует уже озвученные исходники.
Если генерация прервалась, повторите эту ячейку: шард публикуется только целиком.

Для дальнейшего улучшения есть `REAL_WAKE_MANIFEST`: JSONL с действительно
проверенными записями. Строка: `{"wav":"call.wav","label":1,"language":"ru",
"speaker":"person-1","split":"train","text":"бот","reviewed":true}`.
Отрицательные примеры имеют label=0. WAV — mono PCM16, 16 кГц, полное высказывание
короче двух секунд; относительные пути считаются от JSONL. Один человек должен
принадлежать одной части. Не используйте текст, выданный ASR, как истинную метку.


In [ ]:
wake_data.download_models(WORK_DIR, CONFIG)
wake_data.prepare_real(CONFIG)
wake_data.prepare_human(CONFIG)
import time
processes = []
try:
    for rank in range(GPU_COUNT):
        log_path = WORK_DIR / f'tts-gpu-{rank}.log'
        handle = log_path.open('w')
        process = subprocess.Popen([sys.executable, '-u', str(CODE_DIR / 'wake_data_v2.py'),
                                    '--config', str(CONFIG_FILE), '--rank', str(rank), '--world', str(GPU_COUNT)],
                                   stdout=handle, stderr=subprocess.STDOUT)
        processes.append((rank, process, handle, log_path))
    while any(process.poll() is None for _, process, _, _ in processes):
        for rank, process, _, log_path in processes:
            if process.poll() not in (None, 0):
                raise RuntimeError(f'TTS rank {rank} failed; inspect {log_path}')
        statuses = []
        for rank, process, _, log_path in processes:
            lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
            statuses.append(f'GPU {rank}: {lines[-1] if lines else "loading TTS..."}')
        print(' | '.join(statuses), flush=True)
        time.sleep(30)
    for rank, process, _, log_path in processes:
        if process.returncode:
            raise RuntimeError(f'TTS rank {rank} failed; inspect {log_path}')
finally:
    for _, process, handle, _ in processes:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
        handle.close()
rows, dataset_summary = wake_data.merge_manifest(CONFIG, GPU_COUNT)
print(json.dumps(dataset_summary, indent=2))
print('Total windows:', len(rows), '| Source groups:', len({row['source_id'] for row in rows}))


## 5. Проверить примеры и разделение

Все аугментации одного исходного аудио остаются в одной части. Совпадающее
ключевое слово и созвучные негативы намеренно встречаются в разных частях,
но их голоса и исходные записи различаются. Это speaker-held-out split.
Аугментации не считаются независимыми новыми дикторами.
Прослушайте несколько примеров перед большим обучением.


In [ ]:
from IPython.display import Audio, display
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11})
for label in [1, 0]:
    candidates = [row for row in rows if row['split'] == 'train' and row['label'] == label and row['generator'] not in ['real_speech', 'procedural_noise']]
    samples = [next(row for row in candidates if row['generator'] == engine) for engine in TTS_ENGINES
               if any(row['generator'] == engine for row in candidates)]
    for row in samples:
        print(f'label={label}; voice={row["speaker"]}; text={row["text"]}')
        display(Audio(filename=str(WORK_DIR / 'dataset' / row['wav'])))
fig, ax = plt.subplots()
names = list(dataset_summary)
ax.bar(names, [dataset_summary[name]['positive'] for name in names], label='Wake', color='#E69F00', hatch='/')
ax.bar(names, [dataset_summary[name]['negative'] for name in names],
       bottom=[dataset_summary[name]['positive'] for name in names], label='Non-wake', color='#0072B2')
ax.set(ylabel='Synthetic windows (2 seconds)', title='Dataset size by speaker-disjoint split')
ax.legend(); fig.tight_layout(); plt.show()


## 6. Обучение DDP: обе GPU

`torch.distributed.run --standalone --nproc_per_node=2` создаёт два процесса,
`DistributedSampler` даёт каждому свою часть train, DDP синхронизирует градиенты.
Эффективный batch по умолчанию 128 = 64 × 2. Это реальное обучение обеими GPU,
но удвоенная скорость не гарантируется: загрузка данных и синхронизация тоже занимают время.

По умолчанию дообучаются последние два слоя акустического Whisper Tiny encoder
и бинарная голова. Encoder использует learning rate 1e-5, голова — 7e-4.
`temporal_v2` позволяет отдельно сравнить обучение CNN с нуля. Weighted sampler
балансирует классы, русский/английский и TTS-движки; каждые 10 эпох разбирает
трудные примеры **только train**. EMA сглаживает веса. Лучшая эпоха выбирается
по macro recall и худшему языку при заданном validation FPR; loss служит
дополнительным критерием. Максимум 100 эпох, early stopping после 18 без улучшения.
Два seed обучаются последовательно, каждый использует обе GPU через DDP.
Затем validation выбирает одну модель или среднее logits двух. Test не участвует.
После каждой эпохи сохраняются model, optimizer, scheduler, scaler и RNG каждого rank.
Для продолжения загрузите прошлый Output как Dataset, задайте `RESUME_CHECKPOINT`,
сохраните те же данные/seed/число GPU и общее `EPOCHS` исходного запуска.
Для ансамбля задайте путь к `checkpoints/`; в каждом `member-i/` рядом с `latest.pt`
нужны `latest-rng-0.pt` и `latest-rng-1.pt`. Для одной модели — путь к latest.pt.
Смена датасета или количества GPU при resume отклоняется.


In [ ]:
def train_member(member):
    command = [sys.executable, '-m', 'torch.distributed.run', '--standalone',
               f'--nproc_per_node={GPU_COUNT}', str(CODE_DIR / 'wake_train.py'), '--config', str(CONFIG_FILE), '--member', str(member)]
    print(' '.join(command))
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as training_process:
        try:
            for line in training_process.stdout: print(line.rstrip(), flush=True)
            if training_process.wait(): raise subprocess.CalledProcessError(training_process.returncode, command)
        except BaseException:
            training_process.terminate()
            try: training_process.wait(timeout=15)
            except subprocess.TimeoutExpired: training_process.kill(); training_process.wait()
            raise
for member in range(CONFIG['ensemble_members']): train_member(member)
training_run = json.loads((WORK_DIR / 'training_run.json').read_text(encoding='utf-8'))
assert training_run['world_size'] == GPU_COUNT
print(training_run)


## 7. Порог, независимый test и экспорт ONNX

Порог выбирается на **calibration**, чтобы доля false positive среди негативных
окон была не выше заданной цели — и в целом, и отдельно в русском и английском.
После фиксации порога оценивается test. Дополнительные цели FPR 0.2/0.5/1/2%
заданы заранее и показывают компромисс, но основной порог остаётся `TARGET_FPR_PER_WINDOW`.
Число ложных срабатываний, пропуски, recall/precision и группы голосов записываются
в `training_report.json`. FPR на синтетических окнах не является FAR/час реального
непрерывного разговора. При провале recall высокий порог не делает модель хорошей.

Экспорт включает CPU ONNX, веса PyTorch, frontend, конфигурацию, отчёты и источники.
В V4 encoder входит в ONNX; после обучения для инференса нужны только NumPy и ONNX Runtime.
ONNX проверяется сравнением с PyTorch на WAV из test; новый независимый инференс
использует тот же frontend. При `speech_embedding_v3` это отдельные фиксированные
mel/embedding ONNX и обученная бинарная голова; вся цепочка входит в ZIP.


In [ ]:
subprocess.run([sys.executable, str(CODE_DIR / 'wake_export.py'), '--config', str(CONFIG_FILE)], check=True)
MODEL_DIR = WORK_DIR / ('wake-model-SMOKE-ONLY' if SMOKE else 'wake-model')
report = json.loads((MODEL_DIR / 'training_report.json').read_text(encoding='utf-8'))
print('Calibration:', report['calibration'])
print('Untouched test:', report['test'])
if report['calibration']['recall'] < 0.8 or report['test']['recall'] < 0.8:
    print('LOW RECALL: inspect errors and improve data/model before deploying this candidate.')
if report['test']['false_positive_rate_per_window'] > TARGET_FPR_PER_WINDOW:
    print('Test FPR exceeds the calibration target. Do not retune on test; add new training/calibration data.')


## 8. Результаты запуска

Графики показывают фактически выполненное обучение и распределение оценок
независимого синтетического test. Пересечение распределений показывает ошибки;
высокая accuracy на фоне большого числа негативов может скрывать пропуски.
Смотрите одновременно recall и false-positive rate, включая отдельные голоса.


In [ ]:
history = json.loads((WORK_DIR / 'training_history.json').read_text(encoding='utf-8'))
predictions = json.loads((WORK_DIR / 'test_predictions.json').read_text(encoding='utf-8'))
from matplotlib.ticker import MaxNLocator
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for member in sorted({row.get('member', 0) for row in history}):
    subset = [row for row in history if row.get('member', 0) == member]
    axes[0].plot([row['epoch'] for row in subset], [row['train_loss'] for row in subset], label=f'Train seed {member}', linestyle='--')
    axes[0].plot([row['epoch'] for row in subset], [row['validation_loss'] for row in subset], label=f'Validation seed {member}')
    points = [row for row in subset if 'validation_operating_point' in row]
    if points:
        axes[1].plot([row['epoch'] for row in points], [row['validation_operating_point']['macro_recall'] for row in points], label=f'Macro seed {member}')
        axes[1].plot([row['epoch'] for row in points], [row['validation_operating_point']['worst_language_recall'] for row in points], label=f'Worst language seed {member}', linestyle='--')
axes[0].set(title='Loss by epoch (different class weighting)', xlabel='Epoch', ylabel='Binary cross-entropy')
axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
axes[0].legend()
axes[1].set(title=f'Validation recall at FPR ≤ {TARGET_FPR_PER_WINDOW:g}', xlabel='Epoch', ylabel='Recall', ylim=(0, 1))
axes[1].xaxis.set_major_locator(MaxNLocator(integer=True)); axes[1].legend()
for label, title in [(0, 'Non-wake'), (1, 'Wake')]:
    values = [row['probability'] for row in predictions if row['label'] == label]
    axes[2].hist(values, bins=np.linspace(0, 1, 31), histtype='step', linewidth=2, density=True,
                 color='#0072B2' if label == 0 else '#E69F00', label=f'{title} (n={len(values)})')
axes[2].axvline(report['calibration_threshold'], color='black', linestyle='--', label='Calibration threshold')
axes[2].set(title='Held-out test scores', xlabel='Wake probability', ylabel='Density', xlim=(0, 1))
axes[2].legend(); fig.tight_layout()
fig.savefig(MODEL_DIR / 'training_and_scores.png', dpi=150); plt.show()
print('False positives:', report['test']['fp'], '/', report['test']['negative_windows'])
print('Missed wakes:', report['test']['fn'], '/', report['test']['positive_windows'])
print('Language metrics:', json.dumps(report['test_groups']['language'], indent=2))
print('Predeclared FPR profiles:', json.dumps(report.get('threshold_profiles', []), indent=2))
errors = [row for row in predictions if row['predicted_wake'] != bool(row['label'])]
print('First test errors (at most 10):', json.dumps(errors[:10], ensure_ascii=False, indent=2))


## 9. Скачать и проверить в отдельном процессе

В Kaggle Output найдёте ZIP с моделью и `dataset/` с WAV, признаками и manifest.
`INCLUDE_DATASET_IN_ZIP=True` дополнительно упаковывает датасет в отдельный ZIP.
Скачайте `checkpoints/`, если планируете продолжать обучение.
Локальный пример: `python wake_audio.py <model_directory> <mono_16k_pcm16.wav>`.
Для короткой записи frontend дополняет её тишиной слева до двух секунд.
Аудио должно иметь частоту 16 кГц; frontend не угадывает частоту записи.

Обязательно проверьте реальные отдельные обращения, похожие слова и фоновые
разговоры через ваш микрофон. Для оценки непрерывного детектора отдельно нужны
правила окон, debounce/cooldown и длительные реальные негативные записи.
В текущем боте можно будет оставить распознавание команд в Groq и заменить
облачную проверку имени этим локальным детектором после такой проверки.


In [ ]:
import shutil
from IPython.display import FileLink
# A fresh process reloads ONNX and frontend, independent of the training objects.
example = next(row for row in rows if row['split'] == 'test' and row['label'] == 1)
subprocess.run([sys.executable, str(MODEL_DIR / 'wake_audio.py'), str(MODEL_DIR),
                str(WORK_DIR / 'dataset' / example['wav'])], check=True)
for name in ['environment.json', 'source_sha256.json']:
    shutil.copy2(WORK_DIR / name, MODEL_DIR / name)
model_zip = shutil.make_archive(str(MODEL_DIR), 'zip', root_dir=MODEL_DIR)
display(FileLink(model_zip))
if INCLUDE_DATASET_IN_ZIP:
    dataset_zip = shutil.make_archive(str(WORK_DIR / 'synthetic-wake-dataset'), 'zip', root_dir=WORK_DIR / 'dataset')
    display(FileLink(dataset_zip))
print('Model:', model_zip)
print('Dataset:', WORK_DIR / 'dataset' / 'manifest.jsonl')
print('Quality on real users has not been validated; synthetic metrics are in training_report.json.')


## Источники и границы проверки

- [Silero TTS: модели, голоса, standalone API и лицензии](https://github.com/snakers4/silero-models).
  Используются официальные `v5_5_ru` и `v3_en`; SHA-256 загруженных файлов
  записываются в `tts_provenance.json`. Тексты негативов составлены для этого ноутбука.
- [Piper](https://github.com/OHF-Voice/piper1-gpl) и [голоса](https://huggingface.co/rhasspy/piper-voices).
- [MMS/VITS](https://huggingface.co/docs/transformers/model_doc/vits).
- [Kokoro ONNX](https://github.com/thewh1teagle/kokoro-onnx) и [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M).
- [Speech Commands](https://www.tensorflow.org/datasets/catalog/speech_commands): реальные негативные слова,
  исходные ссылки, SHA и лицензионные ссылки сохраняются в `real_provenance.json`.
- [openWakeWord: Google speech embeddings](https://github.com/dscripka/openWakeWord#model-architecture):
  закреплённые преобразования v0.5.1, SHA-256 проверяются до вычисления признаков.
  Код upstream имеет Apache-2.0, для распространяемых готовых моделей upstream
  также указывает CC BY-NC-SA; лицензии и ссылки сохранены в provenance.
- [Whisper Tiny](https://huggingface.co/openai/whisper-tiny): закреплена ревизия
  `169d4a4341b33bc18d8881c4b69c2e104e1cc0af`; оригинальный encoder адаптируется
  к окну 2 секунды и обучается для бинарной задачи, а не генерации транскрипции.
- [PyTorch torchrun](https://pytorch.org/docs/stable/elastic/run.html) и
  [DistributedDataParallel](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html).
- [Kaggle: GPU T4 ×2](https://www.kaggle.com/product-feedback/361104).

Encoder загружается из готовой многоязычной модели, бинарная голова обучается здесь.
Метрики вашего запуска появятся только после выполнения ячеек.
SMOKE-ONLY проверяет код, а не точность; его нельзя считать готовым детектором.
Синтетический датасет воспроизводим по recipe/seed, исходникам и сохранённым
TTS-весам; после обновления пакетов/весов полное совпадение не гарантируется.
Голос aidar и en_20–23 уже встречались в отчёте V1: эта часть test — повторный
benchmark, не полностью новый слепой тест. Новые движки/голоса и реальные
негативы расширяют проверку. Для окончательной оценки нужен новый реальный корпус.
